# VeriClue — Tinh chỉnh PAD trên FAS-VN · FACE-04 · **Người 1 — MiniFASNet**

Notebook này **tự chạy từ đầu đến cuối** bằng nút **Run All** trên Kaggle (GPU T4) hoặc Colab.
Bạn **chỉ sửa `SMOKE`**. Cần **Add Input** output của `prep_pad.ipynb` (Người 1 đã chạy và chia sẻ).
Không cần clone repo: code hệ thống đã được **nhúng sẵn**.

**Notebook làm gì**
1. Cài thư viện, tải trọng số đúng revision, **kiểm SHA-256**.
2. Nạp dữ liệu đã chuẩn bị (`pad_prep`): **không** tự chia lại tập, kiểm SHA-256 file chia tập.
5. Đo MiniFASNet gốc trên `calib_fit` (chỉ notebook MiniFASNet).
6. Dạy mô hình của file này: (a) MiniFASNet V2 dạy toàn bộ, hoặc (b) DINOv2-reg ViT-B/14 dạy vài khối cuối + lớp cuối.
   Dạy trên ảnh cắt đã nén lại + tăng cường; đo trên ảnh cắt **thô đúng `pad.py`**.
7. Chọn checkpoint và **hiệu chỉnh** (calibrator) **chỉ trên `calib_fit`**.
8. Đo test phụ (Axon theo loại tấn công, UniData, Zalo 2022) nếu có.
9. Kiểm cuối: **không bước nào chạm split `test`**.
10. Xuất **một file zip** gửi trưởng nhóm (trưởng nhóm gộp 2 zip bằng `combine_pad.py`).

**Hướng dẫn từng nút bấm:** `ml/notebooks/README-pad.md`.

> Dữ liệu FAS-VN chỉ dùng nghiên cứu: không phân phối lại, không đưa mặt người lên báo cáo/demo công khai, **KHÔNG bật Public**.

## Thông tin môi trường

Ô dưới in phiên bản Python/thư viện, GPU, đĩa trống, RAM. **Khi báo lỗi, gửi kèm output của ô này.**

In [ ]:
# Thông tin môi trường — dán output ô này kèm khi báo lỗi (xem HANDBOOK-pad.md).
# Chỉ đọc phiên bản qua importlib.metadata, KHÔNG import onnxruntime/onnx ở đây (bước 1 có thể cài lại chúng).
import importlib.metadata as _md
import os as _os
import platform as _platform
import shutil as _shutil
import subprocess as _subprocess
import sys as _sys

print("Python", _sys.version.split()[0], "|", _platform.platform())
for _pkg in ("numpy", "torch", "onnxruntime-gpu", "onnxruntime", "onnx", "opencv-python", "opencv-python-headless",
             "transformers", "scikit-learn", "pydantic", "safetensors", "huggingface-hub"):  # fmt: skip
    try:
        print(f"  {_pkg:24s} {_md.version(_pkg)}")
    except _md.PackageNotFoundError:
        print(f"  {_pkg:24s} (chưa cài)")
if _shutil.which("nvidia-smi"):
    _r = _subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total,driver_version",
                          "--format=csv,noheader"], capture_output=True, text=True)  # fmt: skip
    print("  GPU:", _r.stdout.strip() or _r.stderr.strip())
else:
    print("  GPU: không có (Accelerator = None/CPU)")
for _path in ("/kaggle/working", "/content", "."):
    if _os.path.isdir(_path):
        _u = _shutil.disk_usage(_path)
        print(f"  Đĩa {_path}: trống {_u.free / 1e9:.1f} GB / {_u.total / 1e9:.1f} GB")
        break
try:
    with open("/proc/meminfo") as _f:
        _mem = {_l.split(":")[0]: int(_l.split()[1]) for _l in _f}
    print(f"  RAM: trống {_mem['MemAvailable'] / 1e6:.1f} GB / {_mem['MemTotal'] / 1e6:.1f} GB")
except OSError:
    print("  RAM: không đọc được /proc/meminfo (Windows/máy nhóm)")

In [ ]:
# === NGUỒN (sinh bởi scripts/build_pad_notebook.py — KHÔNG sửa tay) ===
SOURCE_COMMIT = '608c260432798a5e7debc554da554e2162bfdb49'
PAD_PY_REL, TUNING_PY_REL = 'ml/src/vericlue_ml/face/pad.py', 'ml/src/vericlue_ml/face/pad_tuning.py'
SOURCE_DIRTY = True  # True = file nguồn có sửa chưa commit lúc sinh notebook
SOURCE_FILES = {
 "ml/src/vericlue_ml/face/pad.py": "2ddebf5434c03c1d91060a55d40d72a78a7d4f380b0988a27326252d5cf186db",
 "ml/src/vericlue_ml/eval/metrics.py": "19fcf605857c0fe5f8936525d582663ce616ef345ab26c4b59901fd716c819b8",
 "ml/src/vericlue_ml/face/pad_tuning.py": "874fbccb7ede8c36973bfdd02b884d1ae224cf1fcc7872c47c3064a0e9ca5fae",
 "ml/src/vericlue_ml/common/calibration.py": "d2ad7ad1458dd4f7b8bad5f41906e6b6d8fbef1fd8dcd9db3b0b4f3ee8b67beb",
 "ml/src/vericlue_ml/common/result.py": "02503101770cd8050d954bf0d19a015ad5f25f2eb8a18bc20221ec2e0e6c51e6",
 "models/manifest.yaml": "34378f6ec4c5352a6e1c8fa9ab6a6bc18a6060f6c0f3936faf41f3b8f7ee55cf"
}
MANIFEST = {
 "arcface-mnet": {
  "repo": "yakhyo/uniface-weights",
  "revision": "bfcc7d48ea8412d55de406cecf96b65b3ca720b2",
  "license": "nghiên cứu phi thương mại",
  "files": [
   {
    "path": "arcface_mnet.onnx",
    "url": "https://huggingface.co/yakhyo/uniface-weights/resolve/bfcc7d48ea8412d55de406cecf96b65b3ca720b2/arcface_mnet.onnx?download=true",
    "sha256": "9cc6e4a75f0e2bf0b1aed94578f144d15175f357bdc05e815e5c4a02b319eb4f"
   }
  ]
 },
 "minifasnet": {
  "repo": "yakhyo/uniface-weights",
  "revision": "bfcc7d48ea8412d55de406cecf96b65b3ca720b2",
  "license": "Apache-2.0",
  "files": [
   {
    "path": "minifasnet_v2.onnx",
    "url": "https://huggingface.co/yakhyo/uniface-weights/resolve/bfcc7d48ea8412d55de406cecf96b65b3ca720b2/minifasnet_v2.onnx?download=true",
    "sha256": "b32929adc2d9c34b9486f8c4c7bc97c1b69bc0ea9befefc380e4faae4e463907"
   }
  ]
 },
 "scrfd-10g": {
  "repo": "yakhyo/uniface-weights",
  "revision": "bfcc7d48ea8412d55de406cecf96b65b3ca720b2",
  "license": "nghiên cứu phi thương mại",
  "files": [
   {
    "path": "det_10g.onnx",
    "url": "https://huggingface.co/yakhyo/uniface-weights/resolve/bfcc7d48ea8412d55de406cecf96b65b3ca720b2/scrfd_10g.onnx?download=true",
    "sha256": "5838f7fe053675b1c7a08b633df49e7af5495cee0493c7dcf6697200b85b5b91"
   }
  ]
 },
 "dinov2-reg-base": {
  "repo": "facebook/dinov2-with-registers-base",
  "revision": "a1d738ccfa7ae170945f210395d99dde8adb1805",
  "license": "Apache-2.0",
  "files": [
   {
    "path": "config.json",
    "url": "https://huggingface.co/facebook/dinov2-with-registers-base/resolve/a1d738ccfa7ae170945f210395d99dde8adb1805/config.json?download=true",
    "sha256": "6af60aa760138fc90db0ba37b7701730b140b9bf1742514412d03050e72bf7e0"
   },
   {
    "path": "preprocessor_config.json",
    "url": "https://huggingface.co/facebook/dinov2-with-registers-base/resolve/a1d738ccfa7ae170945f210395d99dde8adb1805/preprocessor_config.json?download=true",
    "sha256": "14e780d86fa1861f8751f868d7f45425b5feb55c38ca26f152ca5097ab30f828"
   },
   {
    "path": "model.safetensors",
    "url": "https://huggingface.co/facebook/dinov2-with-registers-base/resolve/a1d738ccfa7ae170945f210395d99dde8adb1805/model.safetensors?download=true",
    "sha256": "7a6f7b3b9fa4b8732e707476a03cd6cdce210048582f21aafb7991c17d98e362"
   }
  ]
 }
}
AUX_HF = {
 "axon_display_replay": {
  "repo": "AxonData/Display_replay_attacks",
  "revision": "f5560f659e185ded62c2f09ff502426b0588406e",
  "license": "CC BY 4.0 (thẻ HF)",
  "limit": None
 },
 "axon_print": {
  "repo": "AxonData/print-attack-dataset",
  "revision": "9a3f99429fbc0aefa764313c64e285b94d234acc",
  "license": "CC BY 4.0 (thẻ HF)",
  "limit": 5,
  "video_only": True
 },
 "axon_cutprint": {
  "repo": "AxonData/Anti_Spoofing_Cut_print_attack",
  "revision": "efd7b1cfeb43952057ec4b353a4a9aefdebdc1a3",
  "license": "CC BY 4.0 (thẻ HF)",
  "limit": 5,
  "video_only": True
 },
 "axon_replay_mobile": {
  "repo": "AxonData/Replay_attack_mobile",
  "revision": "5374f9db418da3b287287158b099207892f0fb06",
  "license": "CC BY 4.0 (thẻ HF)",
  "limit": 7,
  "every": 6
 },
 "axon_3dpaper": {
  "repo": "AxonData/3D_paper_mask_attack_dataset_for_Liveness",
  "revision": "b50042b2d78ac7d3cd13fb1baa28a9d77ecef53e",
  "license": "CC BY 4.0 (thẻ HF)",
  "limit": 14,
  "video_only": True,
  "exclude": [
   "144.mp4"
  ]
 },
 "axon_selfies": {
  "repo": "AxonData/face-anti-spoofing-dataset",
  "revision": "89d658fffa10ee93cc3010e53741d435fbadd3f6",
  "license": "CC BY-NC 4.0 (thẻ HF)",
  "limit": None,
  "prefix": "Selfies/"
 },
 "unidata_asian": {
  "repo": "UniqueData/asian-people-liveness-detection-video-dataset",
  "revision": "cc477c4b70e4c8829c7af5c6127fa2b1b69d7c0e",
  "license": "CC BY-NC-ND 4.0 (thẻ HF)",
  "limit": None,
  "prefix": "files/"
 }
}

## Bước 0 — Cấu hình

Thường **không cần sửa gì**. Muốn chạy thử nhanh (~10–15 phút) để kiểm tra mọi thứ trước khi chạy thật,
đổi `SMOKE = False` thành `SMOKE = True` ở dòng đầu ô dưới.
Notebook tự nhận đang ở Kaggle, Colab hay máy nhóm.

In [ ]:
# Bước 0 — CẤU HÌNH. Chỉ sửa những dòng có chú thích "SỬA ĐƯỢC".
import csv
import io
import json
import math
import os
import platform
import random
import re
import shutil
import subprocess
import sys
import time
import urllib.request
from contextlib import contextmanager
from pathlib import Path

SMOKE = False  # SỬA ĐƯỢC: True = chạy thử nhỏ (vài trăm ảnh, 1 epoch)
if os.environ.get("PAD_SMOKE") == "1":  # máy nhóm: chạy thử tự động
    SMOKE = True
SEED = 42  # CẤM đổi: chia tập và tăng cường dựa vào seed này

IN_KAGGLE = Path("/kaggle/input").is_dir()
IN_COLAB = "google.colab" in sys.modules
ENV = "kaggle" if IN_KAGGLE else "colab" if IN_COLAB else "local"
USE_DRIVE = True  # Colab: lưu kết quả + checkpoint vào Google Drive để chạy tiếp khi bị ngắt

if ENV == "kaggle":
    INPUT_ROOT = Path("/kaggle/input")
    OUT_DIR = Path("/kaggle/working")
elif ENV == "colab":
    INPUT_ROOT = Path("/content/data")
    if USE_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")
        OUT_DIR = Path("/content/drive/MyDrive/vericlue_pad")
    else:
        OUT_DIR = Path("/content/vericlue_pad")
else:  # máy nhóm (chạy thử): đường dẫn lấy từ biến môi trường
    INPUT_ROOT = Path(os.environ.get("PAD_INPUT_ROOT", "data/raw"))
    OUT_DIR = Path(os.environ.get("PAD_OUT_DIR", "pad_out"))
FASVN_DIR = os.environ.get("PAD_FASVN_DIR")  # để trống: tự tìm thư mục chứa train_photo/test_photo
LOCAL_MODELS_DIR = os.environ.get("PAD_LOCAL_MODELS_DIR")  # máy nhóm: dùng trọng số sẵn có nếu đúng SHA
AUX_DIRS = [Path(p) for p in os.environ.get("PAD_AUX_DIRS", "").split(os.pathsep) if p]
AUX_FROM_HF = os.environ.get("PAD_AUX_FROM_HF", "0" if ENV == "local" else "1") == "1"
AUX_MAX_FILES = 4 if SMOKE else None  # số file tối đa mỗi bộ test phụ (None = tất cả)

# ── Tham số CHUẨN BỊ DỮ LIỆU: chỉ có tác dụng trong prep_pad.ipynb. CẤM đổi (chỉ trưởng nhóm). ──
# Notebook dạy đọc lại đúng giá trị đã dùng từ pad_prep/prep_done.json.
SMOKE_PER_FOLDER = 100  # chạy thử: số ảnh lấy ngẫu nhiên mỗi thư mục (4 thư mục)
JPEG_QUALITY = 90  # chất lượng JPEG chung khi khử lộ đề
DET_PAD_RATIO = 0.33  # đệm viền đen mỗi cạnh khi SCRFD trượt (ảnh cắt quá sát)
DHASH_DUP = 4  # dHash ≤ 4 và CÙNG nhãn → coi là trùng, giữ 1
# Ngưỡng gom người kiểm trên TOÀN BỘ FAS-VN (20/10, bàn giao FACE-04): ArcFace 0,40 + dHash 8 nối chuỗi thành 1 nhóm
# 4.887/10.878 ảnh; dải cos 0,40–0,45 (5.234 cặp) phần lớn là người khác, dHash 5–8 trên ảnh selfie khác người.
# 0,45 + dHash 4: 6.919 nhóm, lớn nhất 51 ảnh (0,47 %), 100/100 cặp gốc/chụp lại vẫn chung nhóm.
DHASH_LINK = 4  # dHash ảnh ≤ 4 (mọi nhãn) → gom chung nhóm (cặp ảnh gốc / chụp lại)
ARC_THRESHOLD = 0.45  # cosine ArcFace ≥ 0,45 → cùng người (vẫn lỏng, ngay trên ngưỡng bắt đầu nối chuỗi ~0,43)
MAX_GROUP_SHARE = 0.05  # nhóm lớn nhất > 5 % ảnh → dừng (dấu hiệu nối chuỗi)
FRACTIONS = {"train": 0.60, "calib_fit": 0.15, "calib_ltt": 0.10, "test": 0.15}
MAX_BPCER = 0.01  # CẤM đổi: chọn checkpoint/mô hình theo APCER tại BPCER ≤ 1 %
# Cỡ file JPEG SAU khi nén lại phản ánh độ nét/kết cấu của chính điểm ảnh (nội dung), không phải siêu
# dữ liệu; chạy thử 19/10 cho AUC ~0,66–0,73 dù chuẩn hóa cỡ mặt. Mặc định chỉ BÁO, không chặn (xem bàn giao FACE-04).
LEAK_GATE_FILE_SIZE = False

# ── Vai trò của file (sinh tự động) ──
MODEL = "minifasnet"  # CỐ ĐỊNH cho file này (Người 1 — MiniFASNet) — KHÔNG sửa
if MODEL not in ("prep", "minifasnet", "dinov2"):
    raise RuntimeError("File notebook hỏng: MODEL phải là prep, minifasnet hoặc dinov2. Xin trưởng nhóm file mới.")
WORK = OUT_DIR / ("pad_prep" if MODEL == "prep" else "pad_work")
PREP_DIR_ENV = os.environ.get("PAD_PREP_DIR")  # máy nhóm: thư mục pad_prep có sẵn
RUN_MINIFAS = MODEL == "minifasnet"  # phương án (a) + đo MiniFASNet gốc
RUN_DINO = MODEL == "dinov2"  # phương án (b)
# ── Tham số dạy (chỉ trưởng nhóm đổi) ──
EPOCHS_MINIFAS = 1 if SMOKE else 12
EPOCHS_DINO = 1 if SMOKE else 6
LR_MINIFAS = 2e-4
LR_DINO_BLOCKS = 2e-5
LR_DINO_HEAD = 1e-3
DINO_TRAIN_BLOCKS = 2  # số khối transformer cuối được dạy
DINO_SIZE = 224
EXPORT_DINO_ONNX = True  # xuất ONNX cho DINOv2 (lỗi thì vẫn có safetensors)
# ── CHỈ KHI LỖI HẾT BỘ NHỚ (README-pad.md Phần G) ──
BATCH_MINIFAS = 64 if SMOKE else 256  # CUDA out of memory ở bước 6a → 128
BATCH_DINO = 16 if SMOKE else 48  # CUDA out of memory ở bước 6b → 24, rồi 16
NUM_WORKERS = 0 if os.name == "nt" else 2  # hết RAM (notebook tự khởi động lại) → 0
N_BOOT = 50 if SMOKE else 500  # số lần bootstrap theo nhóm người cho khoảng tin cậy

random.seed(SEED)
WORK.mkdir(parents=True, exist_ok=True)
CONFIG = {
    k: v
    for k, v in dict(globals()).items()
    if k.isupper()
    and k not in ("SOURCE_FILES", "MANIFEST", "AUX_HF")
    and isinstance(v, (int, float, str, bool, dict, type(None)))
}
CONFIG.update({"ENV": ENV, "WORK": str(WORK), "INPUT_ROOT": str(INPUT_ROOT)})
PREP_KEYS = ("SMOKE", "SEED", "SMOKE_PER_FOLDER", "JPEG_QUALITY", "DET_PAD_RATIO", "DHASH_DUP",
             "DHASH_LINK", "ARC_THRESHOLD", "FRACTIONS", "DINO_SIZE")  # fmt: skip
PREP_HASH = __import__("hashlib").sha256(
    json.dumps({k: CONFIG[k] for k in PREP_KEYS} | {"src": SOURCE_FILES}, sort_keys=True).encode()
).hexdigest()[:12]

# Thời gian ước tính trên Kaggle T4 khi chạy đầy đủ (~11.400 ảnh). Chạy thử (SMOKE) nhanh hơn nhiều.
ESTIMATES = {
    "1": "3–6 phút (tải ~380 MB trọng số)",
    "2": "1–25 phút (Kaggle đọc dataset lần đầu chậm; lần chạy thật đo 23 phút)",
    "3": "khoảng 80–90 phút trên CPU Kaggle (lần chạy thật đo 81 phút; đọc 8,9 GB ảnh, dò mặt, gom người)",
    "4": "1–2 phút",
    "5": "dưới 1 phút",
    "6a": "10–20 phút",
    "6b": "30–60 phút",
    "7": "dưới 1 phút",
    "8": "5–15 phút (tải ~1,7 GB video test phụ)",
    "9": "dưới 1 phút",
    "10": "1–3 phút",
}
TIMINGS: dict[str, float] = {}


@contextmanager
def step(key: str, title: str):
    est = ESTIMATES.get(key, "?") if not SMOKE else "chạy thử: nhanh"
    print(f"\n===== Bước {key}: {title} — ước tính {est} =====", flush=True)
    started = time.time()
    yield
    TIMINGS[key] = round(time.time() - started, 1)
    print(f"===== Xong bước {key} sau {TIMINGS[key] / 60:.1f} phút =====", flush=True)


# Chạy tiếp sau khi bị ngắt (Kaggle, notebook dạy): nếu đã "Add Input" output của lần chạy trước của
# CHÍNH notebook này, sao chép checkpoint cũ để chạy tiếp.
if ENV == "kaggle" and MODEL != "prep" and not (WORK / "ckpt").exists():
    for pattern in ("*/pad_work/ckpt", "*/*/pad_work/ckpt"):
        found = sorted(INPUT_ROOT.glob(pattern))
        if found:
            print(f"Thấy checkpoint lần chạy trước ở {found[0]} → sao chép để chạy tiếp…")
            shutil.copytree(found[0], WORK / "ckpt", dirs_exist_ok=True)
            break

print(f"Môi trường: {ENV} | MÔ HÌNH: {MODEL} | SMOKE={SMOKE} | thư mục làm việc: {WORK}")
print("Ước tính thời gian từng bước (Kaggle T4, chạy đầy đủ):")
for key, est in ESTIMATES.items():
    print(f"  Bước {key:>3}: {est}")
print("  Tổng: prep_pad khoảng 100–120 phút (CPU); notebook dạy khoảng 1–2,5 giờ (GPU). Notebook in thời gian thật sau mỗi bước.")

## Mã nhúng từ repo (KHÔNG sửa tay)

Bốn ô dưới được `scripts/build_pad_notebook.py` chép nguyên văn từ repo. Muốn đổi thì sửa file gốc trong
repo rồi chạy lại script — để mô hình tinh chỉnh luôn dùng **đúng** tiền xử lý của hệ thống.

In [ ]:
# === NHÚNG TỪ ml/src/vericlue_ml/face/pad.py @ 608c26043279 (sha256 2ddebf5434c03c1d…) ===
# KHÔNG sửa tay: sửa file trong repo rồi chạy scripts/build_pad_notebook.py.
from __future__ import annotations

from collections.abc import Sequence
from dataclasses import dataclass
from typing import Any, Protocol

import numpy as np
from numpy.typing import ArrayLike, NDArray


class PadError(Exception):
    # Bản rút gọn của PadError trong pad.py (notebook không dùng ErrorCode).
    def __init__(self, message: str, code: object = None) -> None:
        super().__init__(message)
        self.code = code


SECOND_FACE_AREA_RATIO = 0.5


PAD_CROP_SCALE = 2.7  # MiniFASNet V2 (uniface DEFAULT_SCALES)


DET_INPUT_SIZE = 640


DET_CONFIDENCE = 0.5


DET_NMS = 0.4


_STRIDES = (8, 16, 32)


_ANCHORS_PER_CELL = 2


@dataclass(frozen=True)
class FaceBox:
    x1: float
    y1: float
    x2: float
    y2: float
    confidence: float
    # 5 mốc (mắt trái, mắt phải, mũi, khóe miệng trái, khóe miệng phải) theo SCRFD; None nếu không có.
    landmarks: tuple[tuple[float, float], ...] | None = None

    @property
    def area(self) -> float:
        return max(0.0, self.x2 - self.x1) * max(0.0, self.y2 - self.y1)


def _letterbox(image: np.ndarray, size: int) -> tuple[np.ndarray, float]:
    import cv2

    height, width = image.shape[:2]
    ratio = height / width
    if ratio > 1.0:
        new_h, new_w = size, int(size / ratio)
    else:
        new_w, new_h = size, int(size * ratio)
    factor = new_h / height
    canvas = np.zeros((size, size, 3), dtype=np.uint8)
    canvas[:new_h, :new_w] = cv2.resize(image, (new_w, new_h))
    return canvas, factor


def _nms(dets: np.ndarray, threshold: float) -> list[int]:
    x1, y1, x2, y2, scores = dets.T
    areas = (x2 - x1 + 1) * (y2 - y1 + 1)
    order = scores.argsort()[::-1]
    keep: list[int] = []
    while order.size > 0:
        i = order[0]
        keep.append(int(i))
        xx1 = np.maximum(x1[i], x1[order[1:]])
        yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]])
        yy2 = np.minimum(y2[i], y2[order[1:]])
        inter = np.maximum(0.0, xx2 - xx1 + 1) * np.maximum(0.0, yy2 - yy1 + 1)
        overlap = inter / (areas[i] + areas[order[1:]] - inter)
        order = order[np.where(overlap <= threshold)[0] + 1]
    return keep


class ScrfdFaceFinder:
    """SCRFD det_10g. Mặc định giữ đúng tiền xử lý của FACE-01 (uniface); GenD (FACE-02) dùng
    `scale=128, swap_rb=True` như `src/retinaface.py` của repo GenD."""

    def __init__(self, session, *, scale: float = 127.5, swap_rb: bool = False) -> None:
        self._scale = scale
        self._swap_rb = swap_rb
        self._session = session
        self._input = session.get_inputs()[0].name
        self._outputs = [o.name for o in session.get_outputs()]
        if len(self._outputs) != 9:
            raise PadError("det_10g.onnx không có 9 đầu ra (3 tầng × điểm, hộp, mốc mặt).")
        self._centers: dict[tuple[int, int], np.ndarray] = {}

    def _anchor_centers(self, cells: int, stride: int) -> np.ndarray:
        key = (cells, stride)
        if key not in self._centers:
            y, x = np.mgrid[:cells, :cells]
            centers = (np.stack((x, y), axis=-1).astype(np.float32) * stride).reshape(-1, 2)
            self._centers[key] = np.tile(centers[:, None, :], (1, _ANCHORS_PER_CELL, 1)).reshape(
                -1, 2
            )
        return self._centers[key]

    def find(self, image: np.ndarray) -> list[FaceBox]:
        canvas, factor = _letterbox(image, DET_INPUT_SIZE)
        if self._swap_rb:
            canvas = np.ascontiguousarray(canvas[:, :, ::-1])
        tensor = ((canvas.astype(np.float32) - 127.5) / self._scale).transpose(2, 0, 1)[None]
        outputs = self._session.run(self._outputs, {self._input: tensor})
        boxes: list[np.ndarray] = []
        points: list[np.ndarray] = []
        scores: list[np.ndarray] = []
        for index, stride in enumerate(_STRIDES):
            score = outputs[index]
            distance = outputs[len(_STRIDES) + index] * stride
            keep = np.where(score >= DET_CONFIDENCE)[0]
            if keep.size == 0:
                continue
            centers = self._anchor_centers(DET_INPUT_SIZE // stride, stride)
            decoded = np.stack(
                [
                    centers[:, 0] - distance[:, 0],
                    centers[:, 1] - distance[:, 1],
                    centers[:, 0] + distance[:, 2],
                    centers[:, 1] + distance[:, 3],
                ],
                axis=-1,
            )
            offsets = outputs[2 * len(_STRIDES) + index] * stride
            marks = np.stack(
                [centers[:, i % 2] + offsets[:, i] for i in range(10)], axis=-1
            ).reshape(-1, 5, 2)
            boxes.append(decoded[keep])
            points.append(marks[keep])
            scores.append(score[keep])
        if not boxes:
            return []
        all_scores = np.vstack(scores)
        order = all_scores.ravel().argsort()[::-1]
        dets = np.hstack((np.vstack(boxes) / factor, all_scores)).astype(np.float32)[order]
        landmarks = (np.vstack(points) / factor)[order]
        keep = _nms(dets, DET_NMS)
        return [
            FaceBox(
                *(float(v) for v in dets[i]),
                landmarks=tuple((float(x), float(y)) for x, y in landmarks[i]),
            )
            for i in keep
        ]


def _softmax(x: np.ndarray) -> np.ndarray:
    e = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e / np.sum(e, axis=-1, keepdims=True)


class MiniFasNetClassifier:
    def __init__(self, session, scale: float = PAD_CROP_SCALE) -> None:
        self._session = session
        self._input = session.get_inputs()[0].name
        self._output = session.get_outputs()[0].name
        height, width = session.get_inputs()[0].shape[2:4]
        self._size = (int(width), int(height))
        self._scale = scale

    def _crop(self, image: np.ndarray, box: FaceBox) -> np.ndarray:
        import cv2

        src_h, src_w = image.shape[:2]
        x, y = int(box.x1), int(box.y1)
        box_w, box_h = int(box.x2 - box.x1), int(box.y2 - box.y1)
        scale = min((src_h - 1) / box_h, (src_w - 1) / box_w, self._scale)
        new_w, new_h = box_w * scale, box_h * scale
        cx, cy = x + box_w / 2, y + box_h / 2
        x1, y1 = max(0, int(cx - new_w / 2)), max(0, int(cy - new_h / 2))
        x2, y2 = min(src_w - 1, int(cx + new_w / 2)), min(src_h - 1, int(cy + new_h / 2))
        return cv2.resize(image[y1 : y2 + 1, x1 : x2 + 1], self._size)

    def p_real(self, image: np.ndarray, box: FaceBox) -> float:
        face = self._crop(image, box).astype(np.float32)  # BGR, không chuẩn hóa
        tensor = np.expand_dims(np.transpose(face, (2, 0, 1)), axis=0)
        logits = self._session.run([self._output], {self._input: tensor})[0]
        return float(_softmax(logits)[0, 1])  # nhãn 1 = thật

In [ ]:
# === NHÚNG TỪ ml/src/vericlue_ml/eval/metrics.py @ 608c26043279 (sha256 19fcf605857c0fe5…) ===
# KHÔNG sửa tay: sửa file trong repo rồi chạy scripts/build_pad_notebook.py.
from sklearn.metrics import roc_auc_score, roc_curve


def _vectors(labels: ArrayLike, probabilities: ArrayLike) -> tuple[NDArray, NDArray]:
    y, p = np.asarray(labels), np.asarray(probabilities, dtype=float)
    if y.ndim != 1 or p.ndim != 1 or y.shape != p.shape or not y.size:
        raise ValueError("Nhãn và xác suất phải là vector không rỗng, cùng chiều.")
    if not np.isin(y, [0, 1]).all() or not np.isfinite(p).all() or np.any((p < 0) | (p > 1)):
        raise ValueError("Nhãn phải là 0/1, xác suất hữu hạn trong [0,1].")
    return y, p


def _threshold(threshold: float) -> None:
    if not np.isfinite(threshold) or not 0 <= threshold <= 1:
        raise ValueError("Ngưỡng xác suất phải nằm trong [0,1].")


def apcer_by_attack(
    labels: ArrayLike,
    probabilities: ArrayLike,
    attack_types: Sequence[str],
    threshold: float,
) -> dict[str, float]:
    """Tỷ lệ tấn công được chấp nhận theo từng attack_type; bỏ lớp thật khỏi mẫu số."""
    y, p = _vectors(labels, probabilities)
    _threshold(threshold)
    attacks = np.asarray(attack_types)
    if attacks.shape != y.shape or any(not isinstance(a, str) for a in attack_types):
        raise ValueError("attack_types phải là vector chuỗi cùng chiều.")
    if any(not str(a).strip() for a in attacks[y == 1]):
        raise ValueError("Mẫu tấn công phải có attack_type.")
    attacks = np.array([str(a).strip() for a in attacks])
    return {
        str(kind): float(np.mean(p[(y == 1) & (attacks == kind)] < threshold))
        for kind in sorted(set(attacks[y == 1]))
    }


def bpcer(labels: ArrayLike, probabilities: ArrayLike, threshold: float) -> float | None:
    """Tỷ lệ người thật bị từ chối; None khi không có mẫu thật."""
    y, p = _vectors(labels, probabilities)
    _threshold(threshold)
    return float(np.mean(p[y == 0] >= threshold)) if np.any(y == 0) else None


def auc(labels: ArrayLike, probabilities: ArrayLike) -> float | None:
    """ROC AUC mô tả; None khi thiếu một lớp."""
    y, p = _vectors(labels, probabilities)
    return float(roc_auc_score(y, p)) if np.unique(y).size == 2 else None


def eer(labels: ArrayLike, probabilities: ArrayLike) -> float | None:
    """EER nội suy tuyến tính trên ROC, chỉ trả chỉ số; không chọn/xuất ngưỡng."""
    y, p = _vectors(labels, probabilities)
    if np.unique(y).size < 2:
        return None
    fpr, tpr = roc_curve(y, p, drop_intermediate=False)[:2]
    difference = 1 - tpr - fpr
    right = int(np.flatnonzero(difference <= 0)[0])
    if difference[right] == 0:
        return float(fpr[right])
    left = right - 1
    fraction = difference[left] / (difference[left] - difference[right])
    return float(fpr[left] + fraction * (fpr[right] - fpr[left]))

In [ ]:
# === NHÚNG TỪ ml/src/vericlue_ml/face/pad_tuning.py @ 608c26043279 (sha256 874fbccb7ede8c36…) ===
# KHÔNG sửa tay: sửa file trong repo rồi chạy scripts/build_pad_notebook.py.
"""FACE-04 — hàm thuần cho notebook tinh chỉnh PAD (`ml/notebooks/prep_pad.ipynb`, `finetune_pad_{minifasnet,dinov2}.ipynb`).

File này được `scripts/build_pad_notebook.py` NHÚNG NGUYÊN VĂN vào notebook (bỏ các dòng
`from vericlue_ml…`, vì notebook nhúng luôn `pad.py`/`eval/metrics.py`). Vì vậy:
- chỉ phụ thuộc numpy/cv2 ở mức module; torch, sklearn, onnx nhập bên trong hàm;
- không đọc file cấu hình repo, không gọi mạng.

Nội dung: kiểm kê định dạng, lọc trùng (SHA-256, dHash), gom người (union-find), chia lại theo
nhóm, chặn đọc split `test`, khử lộ đề (cắt mặt y hệt `pad.py` → JPEG lại, không EXIF) và kiểm
lộ đề bằng mô hình chỉ dùng siêu dữ liệu, chỉ số PAD (APCER tại BPCER ≤ 1 %), tăng cường kiểu
camera trước điện thoại rẻ, và chạy đồ thị ONNX của MiniFASNet bằng torch để dạy lại rồi ghi
trọng số ngược vào đúng đồ thị gốc (đầu vào/ra giữ nguyên).
"""


import hashlib
import math
from collections import defaultdict
from collections.abc import Callable, Iterable, Sequence
from types import SimpleNamespace
from typing import Any

import numpy as np


SPLITS = ("train", "calib_fit", "calib_ltt", "test")
FORBIDDEN_SPLIT = "test"
DEFAULT_FRACTIONS = {"train": 0.60, "calib_fit": 0.15, "calib_ltt": 0.10, "test": 0.15}
LABELS = ("live", "not_live")
LEAK_AUC_MAX = 0.6


# ── kiểm kê ──────────────────────────────────────────────────────────────────────────────
def sniff_format(head: bytes) -> str:
    """Định dạng thật theo byte đầu file (không tin đuôi file)."""
    if head.startswith(b"\xff\xd8\xff"):
        return "jpeg"
    if head.startswith(b"\x89PNG\r\n\x1a\n"):
        return "png"
    if head[:6] in (b"GIF87a", b"GIF89a"):
        return "gif"
    if head[:4] == b"RIFF" and head[8:12] == b"WEBP":
        return "webp"
    if head[:2] == b"BM":
        return "bmp"
    return "unknown"


def extension_mismatch(path: str, fmt: str) -> bool:
    """True nếu đuôi file không khớp định dạng thật (ví dụ PNG mang đuôi .jpg)."""
    ext = path.rsplit(".", 1)[-1].lower() if "." in path else ""
    expected = {"jpeg": {"jpg", "jpeg", "jpe"}, "png": {"png"}, "gif": {"gif"}}
    return ext not in expected.get(fmt, {fmt})


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


# ── trùng lặp ────────────────────────────────────────────────────────────────────────────
def dhash64(gray: np.ndarray) -> int:
    """dHash 64 bit: thu về 9×8, so từng điểm với điểm bên phải."""
    import cv2

    if gray.ndim == 3:
        gray = cv2.cvtColor(gray, cv2.COLOR_BGR2GRAY)
    small = cv2.resize(gray, (9, 8), interpolation=cv2.INTER_AREA).astype(np.int16)
    bits = (small[:, 1:] > small[:, :-1]).ravel()
    return int(sum(1 << i for i, bit in enumerate(bits) if bit))


def _popcount64(values: np.ndarray) -> np.ndarray:
    if hasattr(np, "bitwise_count"):
        return np.bitwise_count(values).astype(np.int64)
    as_bytes = values.view(np.uint8).reshape(*values.shape, 8)
    table = np.array([bin(i).count("1") for i in range(256)], dtype=np.int64)
    return table[as_bytes].sum(axis=-1)


def hamming_pairs(
    hashes: Sequence[int], max_dist: int, block: int = 512
) -> list[tuple[int, int, int]]:
    """Mọi cặp (i, j, khoảng cách) với i < j và khoảng cách Hamming ≤ max_dist."""
    arr = np.asarray([int(h) & ((1 << 64) - 1) for h in hashes], dtype=np.uint64)
    pairs: list[tuple[int, int, int]] = []
    for start in range(0, len(arr), block):
        dist = _popcount64(arr[start : start + block, None] ^ arr[None, :])
        for left, right in zip(*np.nonzero(dist <= max_dist), strict=True):
            i, j = start + int(left), int(right)
            if i < j:
                pairs.append((i, j, int(dist[left, right])))
    return pairs


class UnionFind:
    """Union-find trên chỉ số 0..n-1 (nén đường đi)."""

    def __init__(self, n: int) -> None:
        self.parent = list(range(n))

    def find(self, x: int) -> int:
        root = x
        while self.parent[root] != root:
            root = self.parent[root]
        while self.parent[x] != root:
            self.parent[x], x = root, self.parent[x]
        return root

    def union(self, a: int, b: int) -> None:
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[max(ra, rb)] = min(ra, rb)

    def components(self) -> dict[int, list[int]]:
        out: dict[int, list[int]] = defaultdict(list)
        for i in range(len(self.parent)):
            out[self.find(i)].append(i)
        return dict(out)


def exact_duplicates(shas: Sequence[str], labels: Sequence[str]) -> dict[int, str]:
    """Ảnh phải bỏ vì trùng byte: {chỉ số: lý do}.

    Cùng SHA-256 mà mang 2 nhãn → bỏ TẤT CẢ bản ("sha_conflict"); cùng nhãn → giữ bản có chỉ số
    nhỏ nhất, bỏ phần còn lại ("sha_dup").
    """
    by_sha: dict[str, list[int]] = defaultdict(list)
    for i, sha in enumerate(shas):
        by_sha[sha].append(i)
    drops: dict[int, str] = {}
    for members in by_sha.values():
        if len(members) < 2:
            continue
        if len({labels[i] for i in members}) > 1:
            drops.update({i: "sha_conflict" for i in members})
        else:
            drops.update({i: "sha_dup" for i in sorted(members)[1:]})
    return drops


def near_duplicate_drops(
    hashes: Sequence[int], labels: Sequence[str], candidates: Iterable[int], max_dist: int = 4
) -> dict[int, str]:
    """Gần trùng CÙNG NHÃN (dHash ≤ max_dist): mỗi cụm giữ 1 ảnh (chỉ số nhỏ nhất).

    Cặp gần trùng KHÁC nhãn là cặp "ảnh gốc / ảnh chụp lại màn hình" — giữ cả hai, chỉ gom
    chung nhóm (xem `link_groups`).
    """
    idx = sorted(set(candidates))
    local = [hashes[i] for i in idx]
    uf = UnionFind(len(idx))
    for a, b, _ in hamming_pairs(local, max_dist):
        if labels[idx[a]] == labels[idx[b]]:
            uf.union(a, b)
    drops: dict[int, str] = {}
    for members in uf.components().values():
        for m in sorted(members)[1:]:
            drops[idx[m]] = "dhash_dup"
    return drops


def embedding_pairs(
    embeddings: np.ndarray, threshold: float, block: int = 1024
) -> list[tuple[int, int]]:
    """Cặp (i < j) có cosine ≥ threshold. Hàng NaN (không có mặt) bị bỏ qua."""
    emb = np.asarray(embeddings, dtype=np.float32)
    valid = np.isfinite(emb).all(axis=1)
    index = np.flatnonzero(valid)
    mat = emb[valid]
    norms = np.linalg.norm(mat, axis=1, keepdims=True)
    mat = mat / np.maximum(norms, 1e-12)
    pairs: list[tuple[int, int]] = []
    for start in range(0, len(mat), block):
        sims = mat[start : start + block] @ mat.T
        for left, right in zip(*np.nonzero(sims >= threshold), strict=True):
            i, j = int(index[start + int(left)]), int(index[int(right)])
            if i < j:
                pairs.append((i, j))
    return pairs


def link_groups(keys: Sequence[str], pairs: Iterable[tuple[int, ...]]) -> list[str]:
    """Gom thành phần liên thông ("thà gộp nhầm còn hơn tách sót"); id nhóm ổn định theo keys."""
    uf = UnionFind(len(keys))
    for pair in pairs:
        uf.union(int(pair[0]), int(pair[1]))
    group_of = [""] * len(keys)
    for members in uf.components().values():
        gid = hashlib.sha1("\n".join(sorted(keys[m] for m in members)).encode()).hexdigest()[:16]
        for m in members:
            group_of[m] = gid
    return group_of


def group_size_stats(groups: Sequence[str]) -> dict[str, float]:
    """Phân bố cỡ nhóm người: số nhóm, lớn nhất, tỉ lệ nhóm lớn nhất, p99 — để phát hiện nối chuỗi."""
    sizes: dict[str, int] = defaultdict(int)
    for g in groups:
        sizes[g] += 1
    arr = np.array(sorted(sizes.values()), dtype=float)
    if not arr.size:
        return {"n_groups": 0, "max": 0, "max_share": 0.0, "p99": 0.0, "median": 0.0}
    return {
        "n_groups": int(arr.size),
        "max": int(arr[-1]),
        "max_share": float(arr[-1] / arr.sum()),
        "p99": float(np.percentile(arr, 99)),
        "median": float(np.median(arr)),
    }


# ── chia tập theo nhóm ────────────────────────────────────────────────────────────────────
def assign_splits(
    groups: Sequence[str],
    labels: Sequence[str],
    fractions: dict[str, float] | None = None,
    seed: int = 42,
) -> dict[str, str]:
    """Gán mỗi nhóm vào đúng MỘT split, cân theo số ảnh từng nhãn. Trả {group_id: split}.

    Nhóm lớn xếp trước (trộn ngẫu nhiên theo seed rồi sắp ổn định theo cỡ), mỗi nhóm vào split
    đang thiếu nhiều nhất tính theo số ảnh của từng nhãn trong nhóm.
    """
    fractions = fractions or DEFAULT_FRACTIONS
    if set(fractions) != set(SPLITS) or not math.isclose(sum(fractions.values()), 1.0):
        raise ValueError("fractions phải có đủ 4 split và tổng bằng 1.")
    counts: dict[str, dict[str, int]] = defaultdict(lambda: defaultdict(int))
    totals: dict[str, int] = defaultdict(int)
    for g, lab in zip(groups, labels, strict=True):
        counts[g][lab] += 1
        totals[lab] += 1
    order = sorted(counts)
    rng = np.random.default_rng(seed)
    rng.shuffle(order)
    order.sort(key=lambda g: -sum(counts[g].values()))
    current = {s: defaultdict(float) for s in SPLITS}
    out: dict[str, str] = {}
    for g in order:
        best, best_gain = SPLITS[0], -math.inf
        for s in SPLITS:
            gain = sum(
                (fractions[s] * totals[lab] - current[s][lab]) * n / max(totals[lab], 1)
                for lab, n in counts[g].items()
            )
            if gain > best_gain + 1e-12:
                best, best_gain = s, gain
        out[g] = best
        for lab, n in counts[g].items():
            current[best][lab] += n
    return out


def check_split_integrity(groups: Sequence[str], splits: Sequence[str]) -> None:
    """Mỗi nhóm người chỉ được nằm ở MỘT split; sai thì ném lỗi."""
    seen: dict[str, str] = {}
    for g, s in zip(groups, splits, strict=True):
        if s not in SPLITS:
            raise ValueError(f"Split lạ: {s!r}")
        if seen.setdefault(g, s) != s:
            raise AssertionError(f"Nhóm {g} nằm ở cả {seen[g]} và {s}.")


def assert_not_test(splits: Iterable[str], context: str = "") -> None:
    """Chặn mọi bước của notebook đọc split `test` (chỉ trưởng nhóm chạy test, một lần)."""
    if any(s == FORBIDDEN_SPLIT for s in splits):
        raise AssertionError(
            f"CẤM dùng split 'test' trong notebook ({context}). Test chỉ chạy một lần sau khi chốt."
        )


def rows_of_split(rows: Sequence[dict[str, Any]], split: str) -> list[dict[str, Any]]:
    """Lọc hàng theo split; gọi với 'test' là lỗi."""
    assert_not_test([split], f"rows_of_split({split!r})")
    out = [r for r in rows if r.get("split") == split]
    assert_not_test((r.get("split") for r in out), "rows_of_split")
    return out


# ── mặt và cắt (y hệt pad.py) ────────────────────────────────────────────────────────────
def largest_face(faces: Sequence[FaceBox]) -> tuple[FaceBox | None, str]:
    """Quy tắc của `FacePadDetector.predict`: không mặt / 2 mặt cỡ tương đương / mặt quá nhỏ."""
    if not faces:
        return None, "no_face"
    ordered = sorted(faces, key=lambda f: f.area, reverse=True)
    top = ordered[0]
    if len(ordered) > 1 and ordered[1].area >= SECOND_FACE_AREA_RATIO * top.area:
        return None, "multi_face"
    if top.x2 - top.x1 < 2 or top.y2 - top.y1 < 2:
        return None, "tiny_face"
    return top, "ok"


def shift_clip_box(box: FaceBox, dx: float, dy: float, width: int, height: int) -> FaceBox:
    """Đưa hộp từ ảnh đã đệm viền về toạ độ ảnh gốc và kẹp trong ảnh."""

    def cx(v: float) -> float:
        return float(min(max(v - dx, 0.0), width - 1))

    def cy(v: float) -> float:
        return float(min(max(v - dy, 0.0), height - 1))

    marks = None
    if box.landmarks is not None:
        marks = tuple((x - dx, y - dy) for x, y in box.landmarks)
    return FaceBox(cx(box.x1), cy(box.y1), cx(box.x2), cy(box.y2), box.confidence, marks)


def detect_face(
    finder: Any, image: np.ndarray, pad_ratio: float = 0.33
) -> tuple[FaceBox | None, str, bool, int]:
    """SCRFD như pad.py; trượt thì đệm viền đen `pad_ratio` mỗi cạnh rồi dò lại.

    Trả (hộp trong toạ độ ảnh gốc | None, trạng thái, đã_đệm, số_mặt).
    """
    import cv2

    faces = finder.find(image)
    box, status = largest_face(faces)
    if status != "no_face" or pad_ratio <= 0:
        return box, status, False, len(faces)
    height, width = image.shape[:2]
    px, py = int(round(width * pad_ratio)), int(round(height * pad_ratio))
    padded = cv2.copyMakeBorder(image, py, py, px, px, cv2.BORDER_CONSTANT, value=(0, 0, 0))
    faces = finder.find(padded)
    box, status = largest_face(faces)
    if box is not None:
        box = shift_clip_box(box, px, py, width, height)
        if box.x2 - box.x1 < 2 or box.y2 - box.y1 < 2:
            return None, "tiny_face", True, len(faces)
    return box, status, True, len(faces)


def pad_crop(image: np.ndarray, box: FaceBox, size: int, scale: float = PAD_CROP_SCALE):
    """Cắt mặt bằng ĐÚNG `MiniFasNetClassifier._crop` của pad.py, chỉ đổi cỡ đầu ra."""
    shim = SimpleNamespace(_size=(int(size), int(size)), _scale=scale)
    return MiniFasNetClassifier._crop(shim, image, box)


def arcface_align(image: np.ndarray, landmarks: Sequence[Sequence[float]]) -> np.ndarray:
    """Căn 5 mốc về mẫu ArcFace 112×112 (similarity bình phương tối thiểu, như DATA-03)."""
    import cv2

    target = np.array(
        [
            [38.2946, 51.6963],
            [73.5318, 51.5014],
            [56.0252, 71.7366],
            [41.5493, 92.3655],
            [70.7299, 92.2041],
        ],
        dtype=np.float64,
    )
    x, y = np.asarray(landmarks, dtype=np.float64).T
    design = np.zeros((10, 4), dtype=np.float64)
    design[0::2] = np.column_stack((x, -y, np.ones(5), np.zeros(5)))
    design[1::2] = np.column_stack((y, x, np.zeros(5), np.ones(5)))
    a, b, tx, ty = np.linalg.lstsq(design, target.ravel(), rcond=None)[0]
    transform = np.array([[a, -b, tx], [b, a, ty]], dtype=np.float32)
    return cv2.warpAffine(image, transform, (112, 112), borderValue=0.0)


# ── khử lộ đề ────────────────────────────────────────────────────────────────────────────
def jpeg_roundtrip(image: np.ndarray, quality: int = 90) -> tuple[np.ndarray, bytes]:
    """Nén JPEG lại cùng chất lượng (cv2 không ghi EXIF) rồi giải mã. Trả (ảnh, byte JPEG)."""
    import cv2

    ok, buf = cv2.imencode(".jpg", image, [int(cv2.IMWRITE_JPEG_QUALITY), int(quality)])
    if not ok:
        raise ValueError("Không nén được JPEG.")
    data = buf.tobytes()
    return cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR), data


def meta_features(data: bytes) -> dict[str, float]:
    """Siêu dữ liệu của file ảnh (KHÔNG nhìn điểm ảnh): định dạng, cỡ, EXIF, bảng lượng tử."""
    fmt = sniff_format(data[:16])
    feats = {
        "is_jpeg": float(fmt == "jpeg"),
        "is_png": float(fmt == "png"),
        "file_bytes": float(len(data)),
        "width": 0.0,
        "height": 0.0,
        "has_exif": float(b"Exif\x00\x00" in data[:65536]),
        "n_app_segments": 0.0,
        "qtable_sum": 0.0,
        "progressive": 0.0,
    }
    if fmt == "png" and len(data) >= 24:
        feats["width"] = float(int.from_bytes(data[16:20], "big"))
        feats["height"] = float(int.from_bytes(data[20:24], "big"))
    if fmt != "jpeg":
        return feats
    pos = 2
    while pos + 4 <= len(data):
        if data[pos] != 0xFF:
            break
        marker = data[pos + 1]
        if marker in (0xD8, 0x01) or 0xD0 <= marker <= 0xD7:
            pos += 2
            continue
        length = int.from_bytes(data[pos + 2 : pos + 4], "big")
        body = data[pos + 4 : pos + 2 + length]
        if 0xE0 <= marker <= 0xEF:
            feats["n_app_segments"] += 1
        elif marker == 0xDB and feats["qtable_sum"] == 0 and len(body) >= 65:
            precision = body[0] >> 4
            table = body[1 : 1 + 64 * (2 if precision else 1)]
            feats["qtable_sum"] = float(sum(table))
        elif marker in (0xC0, 0xC1, 0xC2) and len(body) >= 5:
            feats["height"] = float(int.from_bytes(body[1:3], "big"))
            feats["width"] = float(int.from_bytes(body[3:5], "big"))
            feats["progressive"] = float(marker == 0xC2)
        elif marker == 0xDA:
            break
        pos += 2 + length
    return feats


def univariate_auc(values: Sequence[float], labels: Sequence[int]) -> float:
    """max(AUC, 1 − AUC) của một đặc trưng; hằng số → 0,5."""
    x = np.asarray(values, dtype=float)
    y = np.asarray(labels, dtype=int)
    if np.unique(y).size < 2 or np.unique(x).size < 2:
        return 0.5
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty(len(x), dtype=float)
    sorted_x = x[order]
    i = 0
    while i < len(x):
        j = i
        while j + 1 < len(x) and sorted_x[j + 1] == sorted_x[i]:
            j += 1
        ranks[order[i : j + 1]] = (i + j) / 2 + 1
        i = j + 1
    n_pos, n_neg = int(y.sum()), int((1 - y).sum())
    value = (ranks[y == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)
    return float(max(value, 1 - value))


def leak_auc(
    features: Sequence[dict[str, float]],
    labels: Sequence[int],
    groups: Sequence[str],
    seed: int = 42,
    folds: int = 5,
) -> dict[str, Any]:
    """AUC của mô hình CHỈ dùng siêu dữ liệu, kiểm định chéo theo nhóm người.

    - Bỏ cột hằng số; không còn cột nào → AUC = 0,5 (không có thông tin).
    - StratifiedGroupKFold (giữ tỉ lệ nhãn, không tách nhóm người); AUC tính TRÊN TỪNG FOLD rồi lấy
      trung bình, KHÔNG lật max(AUC, 1 − AUC) — tránh thổi phồng do lệch nhãn giữa các fold.
    - Lấy mô hình cao hơn giữa hồi quy logistic và cây tăng cường (chặn thận trọng).
    Kèm AUC đơn biến từng đặc trưng (có lật chiều, chỉ để mô tả).
    """
    from sklearn.ensemble import HistGradientBoostingClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score
    from sklearn.model_selection import StratifiedGroupKFold
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    names = sorted({k for f in features for k in f})
    x = np.array([[f.get(k, 0.0) for k in names] for f in features], dtype=float)
    y = np.asarray(labels, dtype=int)
    g = np.asarray(groups)
    per_feature = {k: univariate_auc(x[:, i], y) for i, k in enumerate(names)}
    varying = [i for i in range(len(names)) if np.unique(x[:, i]).size > 1]
    result: dict[str, Any] = {
        "n": int(len(y)),
        "per_feature": per_feature,
        "models": {},
        "used_features": [names[i] for i in varying],
        "cv": "StratifiedGroupKFold, AUC trung bình theo fold",
    }
    n_splits = min(folds, len(set(g)), int(np.bincount(y, minlength=2).min()) if y.size else 0)
    if np.unique(y).size < 2 or n_splits < 2 or not varying:
        result["model_auc"] = 0.5
        return result
    x = x[:, varying]
    models: dict[str, Callable[[], Any]] = {
        "logistic": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
        "hgb": lambda: HistGradientBoostingClassifier(max_iter=100, random_state=seed),
    }
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    splits = list(splitter.split(x, y, g))
    for name, factory in models.items():
        fold_aucs = []
        for train_idx, test_idx in splits:
            if np.unique(y[train_idx]).size < 2 or np.unique(y[test_idx]).size < 2:
                continue
            model = factory().fit(x[train_idx], y[train_idx])
            fold_aucs.append(
                float(roc_auc_score(y[test_idx], model.predict_proba(x[test_idx])[:, 1]))
            )
        result["models"][name] = float(np.mean(fold_aucs)) if fold_aucs else 0.5
    result["model_auc"] = max(result["models"].values())
    return result


# ── chỉ số PAD (1 = tấn công, điểm ≥ ngưỡng thì từ chối) ─────────────────────────────────
def threshold_at_bpcer(
    labels: Sequence[int], attack_scores: Sequence[float], max_bpcer: float = 0.01
) -> float:
    """Ngưỡng nhỏ nhất sao cho tỉ lệ người thật bị từ chối (điểm ≥ ngưỡng) ≤ max_bpcer."""
    y = np.asarray(labels, dtype=int)
    p = np.asarray(attack_scores, dtype=float)
    live = np.sort(p[y == 0])[::-1]
    if live.size == 0:
        raise ValueError("Cần ít nhất một mẫu thật để chọn ngưỡng.")
    allowed = int(math.floor(max_bpcer * live.size + 1e-9))
    if allowed >= live.size:
        return 0.0
    return float(min(np.nextafter(live[allowed], np.inf), 1.0))


def pad_metrics(
    labels: Sequence[int],
    attack_scores: Sequence[float],
    groups: Sequence[str] | None = None,
    max_bpcer: float = 0.01,
    n_boot: int = 200,
    seed: int = 0,
) -> dict[str, Any]:
    """AUC, EER, ngưỡng tại BPCER ≤ max_bpcer và APCER ở ngưỡng đó; CI 95 % bootstrap theo nhóm."""
    y = np.asarray(labels, dtype=int)
    p = np.clip(np.asarray(attack_scores, dtype=float), 0.0, 1.0)
    t = threshold_at_bpcer(y, p, max_bpcer)
    out: dict[str, Any] = {
        "n_live": int((y == 0).sum()),
        "n_attack": int((y == 1).sum()),
        "auc": auc(y, p),
        "eer": eer(y, p),
        "threshold_at_bpcer": t,
        "bpcer": bpcer(y, p, t),
        "apcer": float(np.mean(p[y == 1] < t)) if np.any(y == 1) else None,
    }
    if groups is None or n_boot <= 0:
        return out
    g = np.asarray(groups)
    unique = np.unique(g)
    out["n_groups"] = int(unique.size)
    members = {k: np.flatnonzero(g == k) for k in unique}
    rng = np.random.default_rng(seed)
    apcers, aucs = [], []
    for _ in range(n_boot):
        pick = rng.choice(unique, size=unique.size, replace=True)
        idx = np.concatenate([members[k] for k in pick])
        yb, pb = y[idx], p[idx]
        if np.unique(yb).size < 2:
            continue
        tb = threshold_at_bpcer(yb, pb, max_bpcer)
        apcers.append(float(np.mean(pb[yb == 1] < tb)))
        aucs.append(float(auc(yb, pb)))
    if apcers:
        out["apcer_ci95"] = [float(np.percentile(apcers, 2.5)), float(np.percentile(apcers, 97.5))]
        out["auc_ci95"] = [float(np.percentile(aucs, 2.5)), float(np.percentile(aucs, 97.5))]
    return out


def selection_key(metrics: dict[str, Any]) -> tuple[float, float, float]:
    """Khóa chọn mô hình/checkpoint trên calib_fit: APCER thấp, rồi EER thấp, rồi AUC cao."""

    def val(name: str, default: float) -> float:
        v = metrics.get(name)
        return default if v is None or not np.isfinite(v) else float(v)

    return (val("apcer", 1.0), val("eer", 1.0), -val("auc", 0.0))


def rank_key(metrics: dict[str, Any]) -> tuple[float, float]:
    """Khóa so sánh đơn lẻ vs gộp (ADR-0015, trưởng nhóm 20/10): APCER@BPCER≤1 % rồi AUC cao."""

    def val(name: str, default: float) -> float:
        v = metrics.get(name)
        return default if v is None or not np.isfinite(v) else float(v)

    return (val("apcer", 1.0), -val("auc", 0.0))


def combo_max(*attack_probabilities: Sequence[float]) -> np.ndarray:
    """Gộp như TrustRouter `fusion.method = max`: xác suất tấn công ĐÃ hiệu chỉnh lớn nhất."""
    if not attack_probabilities:
        raise ValueError("Cần ít nhất một vector xác suất.")
    stacked = np.vstack([np.asarray(p, dtype=float) for p in attack_probabilities])
    return stacked.max(axis=0)


def recommend(singles: dict[str, dict[str, Any]], combo: dict[str, Any] | None) -> dict[str, Any]:
    """Khuyến nghị: COMBO hơn đơn lẻ tốt nhất (APCER, hòa thì AUC) → "dùng cả hai";
    ngược lại dùng đơn lẻ tốt nhất. Trả {choice, best_single, reason_vi}."""
    if not singles:
        raise ValueError("Cần ít nhất một mô hình đơn lẻ.")
    best = min(singles, key=lambda k: rank_key(singles[k]))
    b = singles[best]
    if combo is not None and rank_key(combo) < rank_key(b):
        reason = (
            f"Gộp max thắng: APCER {combo['apcer']:.4f} so với {b['apcer']:.4f} của {best}"
            if combo["apcer"] != b["apcer"]
            else f"Gộp max hòa APCER ({combo['apcer']:.4f}) nhưng AUC cao hơn: "
            f"{combo['auc']:.4f} so với {b['auc']:.4f} của {best}"
        )
        return {"choice": "combo", "best_single": best, "reason_vi": reason}
    if combo is None:
        reason = f"Không có mô hình thứ hai để gộp; {best} tốt nhất."
    else:
        reason = (
            f"Gộp max KHÔNG hơn {best}: APCER {combo['apcer']:.4f} so với {b['apcer']:.4f}, "
            f"AUC {combo['auc']:.4f} so với {b['auc']:.4f} — thêm mô hình thứ hai không đáng."
        )
    return {"choice": best, "best_single": best, "reason_vi": reason}


# ── tăng cường giống camera trước điện thoại rẻ (cùng cho cả 2 lớp) ──────────────────────
def phone_camera_augment(image: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """Lật, tối/thiếu sáng, thu nhỏ rồi phóng lại, mờ, nhiễu cảm biến, nén JPEG 30–95."""
    import cv2

    out = image
    if rng.random() < 0.5:
        out = out[:, ::-1]
    out = out.astype(np.float32)
    if rng.random() < 0.4:  # thiếu sáng (gamma) + lệch độ tương phản
        gamma = rng.uniform(1.2, 2.2)
        out = 255.0 * (out / 255.0) ** gamma
    if rng.random() < 0.5:
        out = out * rng.uniform(0.7, 1.25) + rng.uniform(-20, 20)
    out = np.clip(out, 0, 255).astype(np.uint8)
    h, w = out.shape[:2]
    if rng.random() < 0.3:
        f = rng.uniform(0.35, 0.9)
        small = cv2.resize(
            out, (max(8, int(w * f)), max(8, int(h * f))), interpolation=cv2.INTER_AREA
        )
        out = cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)
    if rng.random() < 0.3:
        k = int(rng.choice([3, 5]))
        out = cv2.GaussianBlur(out, (k, k), 0)
    if rng.random() < 0.4:
        noise = rng.normal(0, rng.uniform(2, 12), out.shape)
        out = np.clip(out.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    if rng.random() < 0.5:
        out, _ = jpeg_roundtrip(np.ascontiguousarray(out), int(rng.integers(30, 96)))
    return np.ascontiguousarray(out)


# ── dữ liệu test phụ ─────────────────────────────────────────────────────────────────────
def aux_label_from_path(dataset: str, rel_path: str) -> tuple[str, str] | None:
    """Nhãn (live|not_live, loại tấn công) của ảnh/video Axon hoặc UniData.

    `dataset` là tên thư mục bộ dữ liệu (ví dụ `axon_print`, `print-attack-dataset`); loại tấn
    công chỉ suy từ tên bộ và tên thư mục, KHÔNG từ tên file (tránh khớp nhầm chuỗi ngẫu nhiên).
    """
    parts = [x for x in rel_path.replace("\\", "/").lower().split("/") if x]
    folders, filename = parts[:-1], (parts[-1] if parts else "")
    key = dataset.lower()
    if any(x in ("real", "live", "selfies", "selfie") for x in folders) or "selfie" in filename:
        return "live", "bona_fide"
    if any(x in key for x in ("unique", "asian", "unidata")):
        return "live", "bona_fide"
    p = "/".join([key, *folders])
    rules = (
        ("cut", "cutout_print"),
        ("3d", "paper_mask_3d"),
        ("mask", "paper_mask_3d"),
        ("print", "print"),
        ("display", "display_replay"),
        ("screen", "display_replay"),
        ("mobile", "mobile_replay"),
        ("replay", "mobile_replay"),
    )
    for word, attack in rules:
        if word in p:
            return "not_live", attack
    return None


def video_frames(path: str, fractions: Sequence[float] = (0.15, 0.5, 0.85)) -> list[np.ndarray]:
    """Lấy vài khung đều nhau trong video (để đo test phụ)."""
    import cv2

    cap = cv2.VideoCapture(str(path))
    frames: list[np.ndarray] = []
    try:
        count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
        if not cap.isOpened() or not math.isfinite(count) or count < 1:
            return frames
        for fraction in fractions:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(max(0, count - 1) * fraction))
            ok, frame = cap.read()
            if ok and frame is not None:
                frames.append(frame)
    finally:
        cap.release()
    return frames


# ── đồ thị ONNX ↔ torch (MiniFASNet: dạy lại mà giữ nguyên đồ thị) ───────────────────────
def onnx_to_ir(model: Any) -> dict[str, Any]:
    """Đọc đồ thị ONNX thành dạng trung gian thuần Python: nút, thuộc tính, initializer."""
    from onnx import helper, numpy_helper

    inits = {t.name: numpy_helper.to_array(t).copy() for t in model.graph.initializer}
    nodes = []
    for node in model.graph.node:
        attrs = {}
        for a in node.attribute:
            value = helper.get_attribute_value(a)
            attrs[a.name] = value.decode() if isinstance(value, bytes) else value
        nodes.append(
            {
                "op": node.op_type,
                "inputs": list(node.input),
                "outputs": list(node.output),
                "attrs": attrs,
            }
        )
    return {
        "inputs": [i.name for i in model.graph.input if i.name not in inits],
        "outputs": [o.name for o in model.graph.output],
        "initializers": inits,
        "nodes": nodes,
    }


def build_torch_graph(ir: dict[str, Any]) -> Any:
    """nn.Module chạy đúng đồ thị IR. Initializer số thực là tham số học được, trừ trung bình
    và phương sai của BatchNormalization (buffer, luôn ở chế độ eval)."""
    import torch
    import torch.nn.functional as F  # noqa: N812
    from torch import nn

    frozen = set()
    for node in ir["nodes"]:
        if node["op"] == "BatchNormalization":
            frozen.update(node["inputs"][3:5])
    supported = {
        "Conv", "PRelu", "Add", "MatMul", "Flatten", "BatchNormalization", "Relu", "Gemm",
        "Mul", "Sigmoid", "GlobalAveragePool", "Reshape",
    }  # fmt: skip
    unknown = {n["op"] for n in ir["nodes"]} - supported
    if unknown:
        raise NotImplementedError(f"Chưa hỗ trợ toán tử ONNX: {sorted(unknown)}")

    class OnnxGraphModule(nn.Module):
        def __init__(self) -> None:
            super().__init__()
            self.ir_nodes = ir["nodes"]
            self.ir_inputs = ir["inputs"]
            self.ir_outputs = ir["outputs"]
            self.key_of: dict[str, str] = {}
            self.params = nn.ParameterDict()
            for i, (name, arr) in enumerate(ir["initializers"].items()):
                key = f"t{i}"
                self.key_of[name] = key
                tensor = torch.from_numpy(np.ascontiguousarray(arr))
                if np.issubdtype(arr.dtype, np.floating) and name not in frozen:
                    self.params[key] = nn.Parameter(tensor.float())
                else:
                    self.register_buffer(key, tensor)

        def _get(self, env: dict[str, Any], name: str) -> Any:
            if name in env:
                return env[name]
            key = self.key_of[name]
            return self.params[key] if key in self.params else getattr(self, key)

        def forward(self, *xs: Any) -> Any:
            env = dict(zip(self.ir_inputs, xs, strict=True))
            for node in self.ir_nodes:
                ins = [self._get(env, n) if n else None for n in node["inputs"]]
                a, op = node["attrs"], node["op"]
                if op == "Conv":
                    pads = list(a.get("pads", [0, 0, 0, 0]))
                    x = ins[0]
                    if pads[0] != pads[2] or pads[1] != pads[3]:
                        x = F.pad(x, (pads[1], pads[3], pads[0], pads[2]))
                        pads = [0, 0, 0, 0]
                    y = F.conv2d(
                        x,
                        ins[1],
                        ins[2] if len(ins) > 2 else None,
                        stride=tuple(a.get("strides", [1, 1])),
                        padding=(pads[0], pads[1]),
                        dilation=tuple(a.get("dilations", [1, 1])),
                        groups=int(a.get("group", 1)),
                    )
                elif op == "PRelu":
                    y = torch.where(ins[0] >= 0, ins[0], ins[0] * ins[1])
                elif op == "Add":
                    y = ins[0] + ins[1]
                elif op == "Mul":
                    y = ins[0] * ins[1]
                elif op == "MatMul":
                    y = torch.matmul(ins[0], ins[1])
                elif op == "Gemm":
                    left = ins[0].t() if a.get("transA", 0) else ins[0]
                    right = ins[1].t() if a.get("transB", 0) else ins[1]
                    y = a.get("alpha", 1.0) * (left @ right)
                    if len(ins) > 2 and ins[2] is not None:
                        y = y + a.get("beta", 1.0) * ins[2]
                elif op == "Flatten":
                    axis = int(a.get("axis", 1))
                    y = ins[0].reshape(int(np.prod(ins[0].shape[:axis])), -1)
                elif op == "BatchNormalization":
                    y = F.batch_norm(
                        ins[0], ins[3], ins[4], ins[1], ins[2], False, 0.0, a.get("epsilon", 1e-5)
                    )
                elif op == "Relu":
                    y = torch.relu(ins[0])
                elif op == "Sigmoid":
                    y = torch.sigmoid(ins[0])
                elif op == "GlobalAveragePool":
                    y = ins[0].mean(dim=(2, 3), keepdim=True)
                else:  # Reshape
                    y = ins[0].reshape([int(v) for v in ins[1].tolist()])
                env[node["outputs"][0]] = y
            outs = [env[n] for n in self.ir_outputs]
            return outs[0] if len(outs) == 1 else tuple(outs)

        def initializer_arrays(self) -> dict[str, np.ndarray]:
            out = {}
            for name, key in self.key_of.items():
                t = self.params[key] if key in self.params else getattr(self, key)
                out[name] = t.detach().cpu().numpy()
            return out

    return OnnxGraphModule()


def write_onnx_weights(model: Any, arrays: dict[str, np.ndarray]) -> Any:
    """Bản sao đồ thị ONNX gốc với initializer thay bằng trọng số mới (giữ kiểu dữ liệu)."""
    import copy

    from onnx import numpy_helper

    new = copy.deepcopy(model)
    for i, tensor in enumerate(new.graph.initializer):
        if tensor.name in arrays:
            old = numpy_helper.to_array(tensor)
            arr = np.asarray(arrays[tensor.name]).astype(old.dtype).reshape(old.shape)
            new.graph.initializer[i].CopyFrom(numpy_helper.from_array(arr, tensor.name))
    return new


def binary_live_loss(logits: Any, is_live: Any, class_weight: Sequence[float] = (1.0, 1.0)) -> Any:
    """CE nhị phân trên đầu 3 lớp của MiniFASNet: lớp 1 = thật, gộp (logsumexp) các lớp còn lại
    thành "giả" — giữ nguyên đầu ra 3 lớp để pad.py đọc softmax[:, 1] như cũ."""
    import torch

    log_p = torch.log_softmax(logits, dim=1)
    log_real = log_p[:, 1]
    others = torch.cat([log_p[:, :1], log_p[:, 2:]], dim=1)
    log_fake = torch.logsumexp(others, dim=1)
    target = is_live.float()
    weight = torch.where(
        target > 0.5,
        torch.full_like(target, float(class_weight[0])),
        torch.full_like(target, float(class_weight[1])),
    )
    loss = -(target * log_real + (1 - target) * log_fake)
    return (weight * loss).sum() / weight.sum()

In [ ]:
# === NHÚNG TỪ ml/src/vericlue_ml/common/calibration.py @ 608c26043279 (sha256 d2ad7ad1458dd4f7…) ===
# KHÔNG sửa tay: sửa file trong repo rồi chạy scripts/build_pad_notebook.py.
# + lớp Module từ ml/src/vericlue_ml/common/result.py
from datetime import UTC, datetime
from enum import StrEnum
from typing import Literal

from pydantic import BaseModel, ConfigDict, Field, model_validator
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler


class Module(StrEnum):
    QUALITY = "quality"
    FACE_PAD = "face_pad"
    FACE_DEEPFAKE = "face_deepfake"
    CHROMAPULSE = "chromapulse"
    CAMPROBE = "camprobe"
    VOICE_SPOOF = "voice_spoof"
    VOICE_CHALLENGE = "voice_challenge"
    LIPSYNC = "lipsync"
    PROVENANCE = "provenance"
    GAZE_TRACKING = "gaze_tracking"
    HEAD_TURN = "head_turn"
    FACE_MATCH = "face_match"


EPSILON = 1e-4


MIN_ISOTONIC_CLASS = 100


MIN_RELATIVE_IMPROVEMENT = 0.02


PLATT_C = 1e6


PLATT_MAX_ITER = 2000


PLATT_TOL = 1e-8


FloatArray = NDArray[np.float64]


class CalibrationError(ValueError):
    """Không được dùng điểm thô thay cho calibrator thiếu/hỏng/sai phiên bản."""


class _Record(BaseModel):
    model_config = ConfigDict(extra="forbid", frozen=True, allow_inf_nan=False)


def _vector(values: ArrayLike, name: str) -> FloatArray:
    result = np.asarray(values, dtype=float)
    if result.ndim != 1 or not result.size or not np.isfinite(result).all():
        raise CalibrationError(f"{name} phải là vector không rỗng, hữu hạn.")
    return result


def _pairs(labels: ArrayLike, probabilities: ArrayLike) -> tuple[FloatArray, FloatArray]:
    y, p = _vector(labels, "labels"), _vector(probabilities, "probabilities")
    if y.shape != p.shape or not np.isin(y, [0, 1]).all():
        raise CalibrationError("Nhãn phải là 0/1 và cùng chiều với xác suất.")
    if np.any((p < 0) | (p > 1)):
        raise CalibrationError("Xác suất phải nằm trong [0, 1].")
    return y, p


class ReliabilityBin(_Record):
    lower: float
    upper: float
    count: int
    mean_probability: float | None
    attack_rate: float | None


def reliability_table(
    labels: ArrayLike, probabilities: ArrayLike, *, n_bins: int = 10
) -> list[ReliabilityBin]:
    """Bin đều [lower, upper); bin cuối gồm 1, giữ bin rỗng với giá trị None."""
    y, p = _pairs(labels, probabilities)
    if isinstance(n_bins, bool) or not isinstance(n_bins, int) or n_bins < 1:
        raise CalibrationError("n_bins phải là số nguyên dương.")
    indices = np.minimum((p * n_bins).astype(int), n_bins - 1)
    rows = []
    for i in range(n_bins):
        mask = indices == i
        count = int(mask.sum())
        rows.append(
            ReliabilityBin(
                lower=i / n_bins,
                upper=(i + 1) / n_bins,
                count=count,
                mean_probability=float(p[mask].mean()) if count else None,
                attack_rate=float(y[mask].mean()) if count else None,
            )
        )
    return rows


def ece(labels: ArrayLike, probabilities: ArrayLike, *, n_bins: int = 10) -> float:
    """ECE nhị phân: trung bình có trọng số |P(tấn công) - tỷ lệ tấn công|."""
    rows = reliability_table(labels, probabilities, n_bins=n_bins)
    total = sum(row.count for row in rows)
    return (
        sum(
            row.count * abs(row.mean_probability - row.attack_rate)
            for row in rows
            if row.count and row.mean_probability is not None and row.attack_rate is not None
        )
        / total
    )


def brier_score(labels: ArrayLike, probabilities: ArrayLike) -> float:
    """Brier nhị phân, không chia thêm cho hai."""
    y, p = _pairs(labels, probabilities)
    return float(np.mean((p - y) ** 2))


def log_loss(labels: ArrayLike, probabilities: ArrayLike, *, epsilon: float = EPSILON) -> float:
    """Log-loss nhị phân với cắt xác suất hữu hạn."""
    y, p = _pairs(labels, probabilities)
    if not 0 < epsilon < 0.5:
        raise CalibrationError("epsilon phải nằm trong (0, 0.5).")
    p = np.clip(p, epsilon, 1 - epsilon)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log1p(-p)))


class CalibrationMetrics(_Record):
    log_loss: float = Field(ge=0)
    ece: float = Field(ge=0, le=1)
    brier: float = Field(ge=0, le=1)


def _metrics(y: FloatArray, p: FloatArray, epsilon: float) -> CalibrationMetrics:
    return CalibrationMetrics(
        log_loss=log_loss(y, p, epsilon=epsilon), ece=ece(y, p), brier=brier_score(y, p)
    )


class CalibFit(_Record):
    """Dữ liệu một module/phiên bản; caller chịu trách nhiệm chọn đúng calib_fit."""

    scores: list[float]
    labels: list[int]
    subject_ids: list[str]
    dataset: str = Field(min_length=1)
    split: Literal["calib_fit"] = "calib_fit"


class Parameters(_Record):
    slope: float | None = None
    intercept: float | None = None
    x: list[float] = Field(default_factory=list)
    y: list[float] = Field(default_factory=list)


class Calibrator(_Record):
    """Artifact JSON, tham số thuần số; không pickle hay tải trọng số mô hình."""

    module: Module
    model_version: str = Field(min_length=1)
    provisional: bool = False
    method: Literal["platt", "isotonic"]
    parameters: Parameters
    epsilon: Literal[0.0001] = EPSILON
    n_pos: int = Field(gt=0)
    n_neg: int = Field(gt=0)
    before: CalibrationMetrics
    after: CalibrationMetrics
    before_reliability: list[ReliabilityBin]
    after_reliability: list[ReliabilityBin]
    date: datetime
    commit: str = Field(min_length=1)
    dataset: str = Field(min_length=1)
    split: Literal["calib_fit"] = "calib_fit"
    cv_folds: int = Field(ge=2)
    cv_log_loss: dict[Literal["platt", "isotonic"], float]
    metrics_scope: Literal["group_oof"] = "group_oof"
    baseline: Literal["clipped_raw_score"] = "clipped_raw_score"

    @model_validator(mode="after")
    def validate_parameters(self) -> Calibrator:
        p = self.parameters
        if self.method == "platt":
            if p.slope is None or p.intercept is None or p.x or p.y or p.slope < 0:
                raise ValueError("Tham số Platt không hợp lệ hoặc ngược chiều.")
        elif (
            p.slope is not None
            or p.intercept is not None
            or not p.x
            or len(p.x) != len(p.y)
            or np.any(np.diff(p.x) <= 0)
            or np.any(np.diff(p.y) < 0)
            or np.any((np.asarray(p.y) < 0) | (np.asarray(p.y) > 1))
        ):
            raise ValueError("Tham số isotonic không hợp lệ.")
        if "platt" not in self.cv_log_loss or any(v < 0 for v in self.cv_log_loss.values()):
            raise ValueError("Thiếu/sai log-loss kiểm định chéo.")
        if self.method == "isotonic" and not _choose_isotonic(
            self.n_pos, self.n_neg, self.cv_log_loss
        ):
            raise ValueError("Isotonic không đạt điều kiện chọn.")
        return self

    @property
    def prior_attack(self) -> float:
        """Tiền nghiệm π trên calib_fit cho FOR-04; không phải tỷ lệ ngoài thực tế."""
        return self.n_pos / (self.n_pos + self.n_neg)

    def predict(self, scores: ArrayLike) -> FloatArray:
        """P(tấn công), luôn được cắt theo epsilon của artifact."""
        return _predict(self.method, self.parameters, _vector(scores, "scores"), self.epsilon)


def _fit_parameters(method: str, x: FloatArray, y: FloatArray) -> Parameters:
    if method == "isotonic":
        model = IsotonicRegression(increasing=True, out_of_bounds="clip").fit(x, y)
        return Parameters(x=model.X_thresholds_.tolist(), y=model.y_thresholds_.tolist())
    scaler = StandardScaler().fit(x.reshape(-1, 1))
    model = LogisticRegression(C=PLATT_C, solver="lbfgs", max_iter=PLATT_MAX_ITER, tol=PLATT_TOL)
    model.fit(scaler.transform(x.reshape(-1, 1)), y)
    slope = float(model.coef_[0, 0] / scaler.scale_[0])
    return Parameters(slope=slope, intercept=float(model.intercept_[0] - slope * scaler.mean_[0]))


def _predict(method: str, p: Parameters, scores: FloatArray, epsilon: float) -> FloatArray:
    if method == "isotonic":
        probabilities = np.interp(scores, p.x, p.y)
    else:
        z = p.slope * scores + p.intercept
        # Stable sigmoid without overflow for extreme raw scores.
        probabilities = np.exp(-np.logaddexp(0, -z))
    return np.clip(probabilities, epsilon, 1 - epsilon)


def _choose_isotonic(n_pos: int, n_neg: int, losses: dict[str, float]) -> bool:
    return (
        min(n_pos, n_neg) >= MIN_ISOTONIC_CLASS
        and "isotonic" in losses
        and losses["isotonic"] <= losses["platt"] * (1 - MIN_RELATIVE_IMPROVEMENT)
    )


def fit(
    calib_fit: CalibFit,
    *,
    module: str | Module,
    model_version: str,
    commit: str,
    n_splits: int = 5,
    epsilon: float = EPSILON,
) -> Calibrator:
    """Chọn bằng GroupKFold OOF log-loss, rồi fit phương pháp thắng trên toàn calib_fit."""
    x = _vector(calib_fit.scores, "scores")
    y, _ = _pairs(calib_fit.labels, np.clip(x, 0, 1))
    groups = np.asarray(calib_fit.subject_ids)
    if groups.shape != x.shape or any(not s.strip() for s in calib_fit.subject_ids):
        raise CalibrationError("subject_ids phải không rỗng và cùng chiều với scores.")
    if np.unique(y).size != 2:
        raise CalibrationError("calib_fit phải có cả hai lớp.")
    if not isinstance(n_splits, int) or isinstance(n_splits, bool) or n_splits < 2:
        raise CalibrationError("GroupKFold cần ít nhất hai phần.")
    if epsilon != EPSILON:
        raise CalibrationError("Fit phải dùng epsilon=1e-4.")
    if roc_auc_score(y, x) < 0.5:
        raise CalibrationError("AUC < 0.5: nghi score ngược chiều tấn công.")
    folds = min(n_splits, np.unique(groups).size)
    if folds < 2:
        raise CalibrationError("GroupKFold cần ít nhất hai subject.")
    splits = list(GroupKFold(n_splits=folds).split(x, y, groups))
    if any(np.unique(y[train]).size != 2 for train, _ in splits):
        raise CalibrationError("Một fold fit thiếu lớp; cần thêm subject hoặc đổi n_splits.")
    n_pos, n_neg = int(y.sum()), int((1 - y).sum())
    methods = ["platt", "isotonic"] if min(n_pos, n_neg) >= MIN_ISOTONIC_CLASS else ["platt"]
    losses: dict[str, float] = {}
    predictions: dict[str, FloatArray] = {}
    for method in methods:
        oof = np.empty_like(x)
        for train, valid in splits:
            parameters = _fit_parameters(method, x[train], y[train])
            oof[valid] = _predict(method, parameters, x[valid], epsilon)
        predictions[method] = oof
        losses[method] = log_loss(y, oof, epsilon=epsilon)
    method = "isotonic" if _choose_isotonic(n_pos, n_neg, losses) else "platt"
    return Calibrator(
        module=Module(module),
        model_version=model_version,
        method=method,
        parameters=_fit_parameters(method, x, y),
        epsilon=epsilon,
        n_pos=n_pos,
        n_neg=n_neg,
        before=_metrics(y, np.clip(x, epsilon, 1 - epsilon), epsilon),
        after=_metrics(y, predictions[method], epsilon),
        before_reliability=reliability_table(y, np.clip(x, epsilon, 1 - epsilon)),
        after_reliability=reliability_table(y, predictions[method]),
        date=datetime.now(UTC),
        commit=commit,
        dataset=calib_fit.dataset,
        cv_folds=folds,
        cv_log_loss=losses,
    )


calibration_fit = fit  # tránh trùng tên; fit = calibration.fit của hệ thống

## Bước 1 — Cài thư viện, tải trọng số đúng revision và kiểm SHA-256

Kaggle/Colab đã có sẵn PyTorch. Ô này chỉ cài thêm vài gói nhỏ (phiên bản ghim) và tải trọng số từ
Hugging Face. Mỗi file được kiểm SHA-256 theo `models/manifest.yaml` của repo — sai là dừng.

In [ ]:
with step("1", "Cài thư viện và tải trọng số"):

    def pip(*args: str) -> None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

    def probe(code: str) -> str:
        r = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True)
        return r.stdout.strip() if r.returncode == 0 else "ERR"

    if ENV != "local":
        pip("onnx==1.23.2", "safetensors==0.8.0")
        has_cuda = "import onnxruntime as o; print('CUDAExecutionProvider' in o.get_available_providers())"
        if MODEL != "prep" and probe(has_cuda) != "True":  # prep chạy CPU, không cần onnxruntime-gpu
            subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "onnxruntime"])
            pip("onnxruntime-gpu==1.31.0")
        if MODEL == "prep" and probe("import onnxruntime; print(1)") != "1":
            pip("onnxruntime==1.31.0")
        if probe("from transformers import Dinov2WithRegistersModel; print(1)") != "1":
            pip("transformers==4.57.6")
        if ENV == "colab":
            pip("kaggle==1.7.4.5")

    import cv2
    import numpy as np
    import onnx
    import sklearn
    import torch
    import transformers
    import onnxruntime as ort

    if hasattr(ort, "preload_dlls"):
        try:
            ort.preload_dlls()  # dùng thư viện CUDA đi kèm PyTorch
        except Exception as exc:  # noqa: BLE001
            print("preload_dlls:", exc)
    torch.manual_seed(SEED)
    torch.backends.cuda.matmul.allow_tf32 = False  # số khớp ONNX/CPU khi xuất (T4 không có TF32)
    torch.backends.cudnn.allow_tf32 = False
    np.random.seed(SEED)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    VERSIONS = {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "opencv": cv2.__version__,
        "torch": torch.__version__,
        "onnx": onnx.__version__,
        "onnxruntime": ort.__version__,
        "transformers": transformers.__version__,
        "sklearn": sklearn.__version__,
        "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "none",
    }
    print(json.dumps(VERSIONS, indent=1))
    if DEVICE == "cpu" and MODEL != "prep":
        print("CẢNH BÁO: KHÔNG THẤY GPU. Kaggle: Settings → Accelerator → GPU T4 x2 (cần xác minh số điện thoại).")

    WEIGHTS = WORK / "weights"

    def sha256_file(path: Path) -> str:
        digest = __import__("hashlib").sha256()
        with open(path, "rb") as handle:
            for block in iter(lambda: handle.read(1 << 20), b""):
                digest.update(block)
        return digest.hexdigest()

    def fetch(model_id: str, filename: str) -> Path:
        entry = MANIFEST[model_id]
        spec = next(f for f in entry["files"] if f["path"] == filename)
        if LOCAL_MODELS_DIR:
            local = Path(LOCAL_MODELS_DIR) / model_id / filename
            if local.is_file() and sha256_file(local) == spec["sha256"]:
                print(f"  OK {model_id}/{filename} (có sẵn trên máy) sha256={spec['sha256']}")
                return local
        dest = WEIGHTS / model_id / filename
        if not dest.is_file() or sha256_file(dest) != spec["sha256"]:
            dest.parent.mkdir(parents=True, exist_ok=True)
            part = dest.with_name(dest.name + ".part")
            for attempt in range(1, 4):
                try:
                    print(f"  tải {model_id}/{filename} (lần {attempt})…", flush=True)
                    urllib.request.urlretrieve(spec["url"], part)
                    break
                except Exception as exc:  # noqa: BLE001
                    print("   lỗi tải:", exc)
                    time.sleep(10 * attempt)
            else:
                raise RuntimeError(
                    f"KHÔNG TẢI ĐƯỢC {model_id}/{filename} từ Hugging Face. Kiểm tra Settings → "
                    "Internet = On, rồi chạy lại. Nếu HF đang lỗi, thử lại sau 15 phút."
                )
            part.replace(dest)
        digest = sha256_file(dest)
        if digest != spec["sha256"]:
            dest.unlink()
            raise RuntimeError(f"SAI SHA-256 {model_id}/{filename}: {digest} ≠ {spec['sha256']}")
        print(f"  OK {model_id}/{filename} rev={entry['revision'][:8]} sha256={digest}")
        return dest

    PATH_MINIFAS = fetch("minifasnet", "minifasnet_v2.onnx")
    PATH_SCRFD = fetch("scrfd-10g", "det_10g.onnx")
    PATH_ARCFACE = fetch("arcface-mnet", "arcface_mnet.onnx")
    DINO_DIR = None
    if RUN_DINO:
        for name in ("config.json", "preprocessor_config.json", "model.safetensors"):
            DINO_DIR = fetch("dinov2-reg-base", name).parent

    def make_session(path: Path, cpu_only: bool = False):
        options = ort.SessionOptions()
        options.log_severity_level = 3
        if cpu_only:  # chuẩn bị dữ liệu: CPU để mọi máy ra cùng kết quả
            options.intra_op_num_threads = min(4, os.cpu_count() or 4)  # quá nhiều luồng làm CPU chậm hẳn
            options.inter_op_num_threads = 1
            return ort.InferenceSession(str(path), options, providers=["CPUExecutionProvider"])
        if "CUDAExecutionProvider" in ort.get_available_providers():
            try:
                session = ort.InferenceSession(
                    str(path), options, providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
                )
                if "CUDAExecutionProvider" in session.get_providers():
                    return session
            except Exception as exc:  # noqa: BLE001
                print("  ONNX CUDA lỗi, dùng CPU:", str(exc)[:200])
        return ort.InferenceSession(str(path), options, providers=["CPUExecutionProvider"])

    SCRFD_SESSION = make_session(PATH_SCRFD, cpu_only=MODEL == "prep")
    ARC_SESSION = make_session(PATH_ARCFACE, cpu_only=MODEL == "prep")
    MINIFAS_SESSION = make_session(PATH_MINIFAS)
    FINDER = ScrfdFaceFinder(SCRFD_SESSION)  # đúng lớp của pad.py (chuẩn hóa uniface)
    MINI_SIZE = int(MINIFAS_SESSION.get_inputs()[0].shape[2])
    print("ONNX Runtime dùng:", SCRFD_SESSION.get_providers()[0], "| MiniFASNet vào", MINI_SIZE, "px")
    if SCRFD_SESSION.get_providers()[0] == "CPUExecutionProvider" and not SMOKE and MODEL != "prep":
        print("LƯU Ý: dò mặt chạy CPU nên bước 3 có thể mất 80–90 phút. Vẫn chạy được, cứ để chạy.")

## Bước 2 — Nạp dữ liệu đã chuẩn bị (output của `prep_pad.ipynb`)

Notebook này **không** tự làm sạch hay chia tập: nó dùng đúng output Người 1 đã chạy bằng `prep_pad.ipynb`
(Kaggle: Add Input → tab Notebooks/Your Work hoặc notebook được chia sẻ). Nhờ vậy hai người dùng **cùng một** cách chia.

In [ ]:
with step("2", "Nạp dữ liệu đã chuẩn bị"):
    def find_prep():
        if PREP_DIR_ENV:
            return Path(PREP_DIR_ENV)
        roots = [INPUT_ROOT, OUT_DIR] + ([Path("/content/drive/MyDrive")] if ENV == "colab" else [])
        for root in roots:
            for pattern in ("pad_prep/prep_done.json", "*/pad_prep/prep_done.json", "*/*/pad_prep/prep_done.json"):
                hits = sorted(root.glob(pattern)) if root.is_dir() else []
                if hits:
                    return hits[0].parent
        return None

    PREP_DIR = find_prep()
    if PREP_DIR is None or not (PREP_DIR / "prep_done.json").is_file():
        raise RuntimeError(
            "KHÔNG TÌM THẤY DỮ LIỆU ĐÃ CHUẨN BỊ (thư mục pad_prep). Kaggle: cột phải → + Add Input → tab "
            "'Notebooks' (hoặc 'Your Work') → tìm notebook prep_pad mà Người 1 đã chia sẻ → Add, rồi Run All lại. "
            "Colab: thêm lối tắt thư mục vericlue_pad (Người 1 chia sẻ) vào My Drive. Chưa có thì hỏi Người 1 / trưởng nhóm."
        )
    PREP = json.loads((PREP_DIR / "prep_done.json").read_text())
    for rel in (PAD_PY_REL, TUNING_PY_REL):
        if PREP["source_files_sha256"].get(rel) != SOURCE_FILES[rel]:
            raise RuntimeError(
                f"Dữ liệu chuẩn bị được tạo bằng phiên bản code KHÁC ({rel}). Xin trưởng nhóm bộ notebook mới cùng lúc "
                "và cho Người 1 chạy lại prep_pad.ipynb."
            )
    if PREP["smoke"] and not SMOKE:
        raise RuntimeError(
            "Dữ liệu chuẩn bị là bản CHẠY THỬ (SMOKE) — không dạy thật trên đó. Đợi Người 1 chạy prep_pad.ipynb với SMOKE = False."
        )
    globals().update(PREP["config"])  # dùng đúng tham số chuẩn bị (seed, ngưỡng gom người, JPEG…)
    INT_COLS = ("ext_mismatch", "decode_ok", "det_padded", "n_faces", "img_w", "img_h")
    with open(PREP_DIR / "rows.csv", newline="", encoding="utf-8") as handle:
        rows = list(csv.DictReader(handle))
    for r in rows:
        for k, v in list(r.items()):
            if k in INT_COLS:
                r[k] = int(v) if v != "" else 0
            elif k.startswith(("o_", "m80_", "m224_", "box_")):
                r[k] = float(v) if v != "" else float("nan")
    N = len(rows)
    SPLIT_FILE_SHA = {name: sha256_file(PREP_DIR / name) for name in PREP["split_files_sha256"]}
    if SPLIT_FILE_SHA != PREP["split_files_sha256"]:
        raise RuntimeError("File chia tập trong pad_prep bị sửa (SHA-256 không khớp prep_done.json). Báo trưởng nhóm.")
    SPLITS_SHA = SPLIT_FILE_SHA["splits.csv"]
    CLEAN_REPORT = json.loads((PREP_DIR / "cleaning_report.json").read_text())
    LEAK = json.loads((PREP_DIR / "leak_check.json").read_text())
    CROPS80, CROPS224 = (np.load(PREP_DIR / f, mmap_mode="r") for f in ("crops80.npy", "crops224.npy"))
    RAW80, RAW224 = (np.load(PREP_DIR / f, mmap_mode="r") for f in ("raw80.npy", "raw224.npy"))
    if CROPS80.shape[1] != MINI_SIZE or CROPS224.shape[1] != DINO_SIZE or len(CROPS80) != N:
        raise RuntimeError("Ảnh cắt trong pad_prep không đúng cỡ. Báo trưởng nhóm.")
    print(f"Dùng dữ liệu chuẩn bị ở {PREP_DIR} (commit {PREP['source_commit'][:12]}, {N} ảnh, SMOKE={PREP['smoke']})")
    print(f"splits.csv SHA-256 = {SPLITS_SHA}  (Người 1 và Người 2 phải giống nhau)")
    print(f"Kiểm lộ đề lúc chuẩn bị: AUC siêu dữ liệu = {LEAK['model_auc']:.3f} (đạt ≤ {LEAK_AUC_MAX})")

## Bước 5 — Đo MiniFASNet GỐC (trước tinh chỉnh) trên calib_fit

Điểm tấn công = 1 − p_real (giống `score` của hệ thống). Chỉ số: AUC, EER, ngưỡng tại BPCER ≤ 1 % và
APCER ở ngưỡng đó; khoảng tin cậy 95 % bằng bootstrap **theo nhóm người**.

In [ ]:
with step("5", "Đo MiniFASNet gốc trên calib_fit"):
    USED: set[int] = set()  # mọi chỉ số ảnh đã được đọc để dạy/đo — bước 9 kiểm không có ảnh test

    def take_idx(split: str) -> list[int]:
        idx = [i for i, r in enumerate(rows) if r["status"] == "kept" and r["split"] == split]
        assert_not_test([split] + [rows[i]["split"] for i in idx], f"take_idx({split})")
        USED.update(idx)
        return idx

    IDX = {"train": take_idx("train"), "calib_fit": take_idx("calib_fit"), "calib_ltt": take_idx("calib_ltt")}
    LIVE = {s: np.array([int(rows[i]["label"] == "live") for i in idx]) for s, idx in IDX.items()}
    GROUPS = {s: [rows[i]["group_id"] for i in idx] for s, idx in IDX.items()}

    def onnx_p_live(session, idx, batch: int = 256) -> np.ndarray:
        name_in, name_out = session.get_inputs()[0].name, session.get_outputs()[0].name
        out = []
        for s in range(0, len(idx), batch):
            x = np.asarray(RAW80[idx[s : s + batch]]).astype(np.float32).transpose(0, 3, 1, 2)  # đúng pad.py: BGR, 0–255
            out.append(_softmax(session.run([name_out], {name_in: x})[0])[:, 1])  # nhãn 1 = thật
        return np.concatenate(out)

    def evaluate(p_live: np.ndarray, n_boot: int = N_BOOT) -> dict:
        return pad_metrics(1 - LIVE["calib_fit"], 1 - p_live, GROUPS["calib_fit"], MAX_BPCER, n_boot, SEED)

    SCORES: dict = {}  # p_live trên calib_fit (dùng để chọn và hiệu chỉnh)
    SCORES_LTT: dict = {}  # p_live trên calib_ltt (CHỈ để xuất cho ROUT-03, không nhìn ở đây)
    METRICS: dict = {}
    if RUN_MINIFAS:
        SCORES["baseline"] = onnx_p_live(MINIFAS_SESSION, IDX["calib_fit"])
        SCORES_LTT["baseline"] = onnx_p_live(MINIFAS_SESSION, IDX["calib_ltt"])
        METRICS["baseline"] = evaluate(SCORES["baseline"])
    HISTORY: dict[str, list] = {}

    def show(name: str, m: dict) -> None:
        ci = m.get("apcer_ci95")
        ci_txt = f" [{ci[0]:.3f}–{ci[1]:.3f}]" if ci else ""
        print(f"  {name:16s} AUC={m['auc']:.4f} EER={m['eer']:.4f} APCER@BPCER≤{MAX_BPCER:.0%}={m['apcer']:.4f}{ci_txt} "
              f"(ngưỡng {m['threshold_at_bpcer']:.4f}, BPCER {m['bpcer']:.4f}; live {m['n_live']}, tấn công {m['n_attack']})")

    if "baseline" in METRICS:
        show("MiniFASNet gốc", METRICS["baseline"])
    else:
        print("  (Chạy DINOv2: MiniFASNet gốc do Người 1 đo.)")

## Bước 6 — Tinh chỉnh hai phương án

Tăng cường "camera trước điện thoại rẻ" (tối, nhiễu, mờ, thu nhỏ, JPEG 30–95, lật) áp cho **cả hai lớp**.
Sau mỗi epoch lưu checkpoint vào `pad_work/ckpt/` — nếu bị ngắt, chạy lại notebook sẽ **tự chạy tiếp**.

- **(a) MiniFASNet V2**: chạy đúng đồ thị ONNX gốc bằng PyTorch, dạy toàn bộ trọng số; đầu vào giữ y hệt
  `pad.py` (ảnh BGR 80×80, 0–255, không chuẩn hóa), đầu ra 3 lớp, lớp 1 = thật.
- **(b) DINOv2-reg ViT-B/14** (`facebook/dinov2-with-registers-base`, Apache-2.0): cùng vùng cắt mặt,
  224×224, chuẩn hóa ImageNet; chỉ dạy vài khối cuối + lớp cuối.

In [ ]:
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, Dataset

MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def to_input(img: np.ndarray, kind: str):
    if kind == "minifas":  # y hệt MiniFasNetClassifier.p_real: BGR float32 0–255, NCHW
        return torch.from_numpy(np.ascontiguousarray(img.transpose(2, 0, 1)).astype(np.float32))
    rgb = (img[:, :, ::-1].astype(np.float32) / 255.0 - MEAN) / STD
    return torch.from_numpy(np.ascontiguousarray(rgb.transpose(2, 0, 1)))


class CropSet(Dataset):
    def __init__(self, kind: str, idx: list[int], live: np.ndarray, augment: bool) -> None:
        assert_not_test((rows[i]["split"] for i in idx), "CropSet")
        USED.update(idx)
        self.kind, self.idx, self.live, self.augment = kind, list(idx), live, augment
        # Dạy: ảnh đã nén lại JPEG (khử lộ đề) + tăng cường. Đo/hiệu chỉnh: ảnh cắt thô đúng pad.py.
        size = "80" if kind == "minifas" else "224"
        self.path = PREP_DIR / (f"crops{size}.npy" if augment else f"raw{size}.npy")
        self.arr = None
        self.epoch = 0

    def __len__(self) -> int:
        return len(self.idx)

    def __getitem__(self, k: int):
        if self.arr is None:
            self.arr = np.load(self.path, mmap_mode="r")
        img = np.array(self.arr[self.idx[k]])
        if self.augment:
            img = phone_camera_augment(img, np.random.default_rng((SEED, self.epoch, int(self.idx[k]))))
        return to_input(img, self.kind), int(self.live[k])


def trainable_state(model) -> dict:
    names = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in names}


def predict_p_live(model, kind: str, idx: list[int], batch: int) -> np.ndarray:
    loader = DataLoader(CropSet(kind, idx, np.zeros(len(idx)), False), batch_size=batch, num_workers=NUM_WORKERS)
    model.eval()
    out = []
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=(kind == "dino" and DEVICE == "cuda")):
        for x, _ in loader:
            out.append(torch.softmax(model(x.to(DEVICE)).float(), dim=1)[:, 1].cpu().numpy())
    return np.concatenate(out)


def train_model(name: str, model, groups_params, kind: str, epochs: int, batch: int, loss_fn) -> dict:
    ckdir = WORK / "ckpt" / name
    ckdir.mkdir(parents=True, exist_ok=True)
    last, best = ckdir / "last.pt", ckdir / "best.pt"
    data = CropSet(kind, IDX["train"], LIVE["train"], True)
    gen = torch.Generator().manual_seed(SEED)
    loader = DataLoader(data, batch_size=batch, shuffle=True, drop_last=len(data) > batch,
                        num_workers=NUM_WORKERS, generator=gen)  # fmt: skip
    opt = torch.optim.AdamW(groups_params, weight_decay=1e-4)
    total = max(1, epochs * len(loader))
    warm = max(1, len(loader) // 2)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(s, total) / total))
    )
    use_amp = kind == "dino" and DEVICE == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    state = {"epoch": 0, "history": [], "best_key": None, "best_epoch": None}
    if last.exists():
        ck = torch.load(last, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck["model"], strict=False)
        opt.load_state_dict(ck["opt"])
        sched.load_state_dict(ck["sched"])
        scaler.load_state_dict(ck["scaler"])
        state = ck["state"]
        print(f"  Chạy tiếp {name} từ checkpoint: đã xong {state['epoch']}/{epochs} epoch")
    for epoch in range(state["epoch"], epochs):
        data.epoch = epoch
        model.train()
        t0, seen, loss_sum = time.time(), 0, 0.0
        for step_i, (x, y) in enumerate(loader, 1):
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE)
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                logits = model(x)
            loss = loss_fn(logits.float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            seen += len(y)
            loss_sum += loss.item() * len(y)
            if step_i % 50 == 0:
                rate = seen / (time.time() - t0)
                print(f"    {name} epoch {epoch + 1} bước {step_i}/{len(loader)} loss {loss_sum / seen:.4f} "
                      f"({rate:.0f} ảnh/giây, epoch còn ~{(len(data) - seen) / rate / 60:.1f} phút)", flush=True)
        p = predict_p_live(model, kind, IDX["calib_fit"], batch * 2)
        m = evaluate(p, n_boot=0)
        record = {"epoch": epoch + 1, "loss": loss_sum / max(seen, 1), "seconds": round(time.time() - t0, 1),
                  **{k: m[k] for k in ("auc", "eer", "apcer", "bpcer", "threshold_at_bpcer")}}  # fmt: skip
        state["history"].append(record)
        key = list(selection_key(m))
        if state["best_key"] is None or key < state["best_key"]:
            state["best_key"], state["best_epoch"] = key, epoch + 1
            torch.save(trainable_state(model), best)
        state["epoch"] = epoch + 1
        torch.save({"model": trainable_state(model), "opt": opt.state_dict(), "sched": sched.state_dict(),
                    "scaler": scaler.state_dict(), "state": state}, last)  # fmt: skip
        print(f"  {name} epoch {epoch + 1}/{epochs}: loss {record['loss']:.4f} | calib_fit AUC {m['auc']:.4f} "
              f"EER {m['eer']:.4f} APCER@BPCER≤1% {m['apcer']:.4f} | {record['seconds'] / 60:.1f} phút", flush=True)
    model.load_state_dict(torch.load(best, map_location=DEVICE), strict=False)
    print(f"  {name}: chọn checkpoint epoch {state['best_epoch']} (theo calib_fit)")
    return state


n_live, n_attack = int(LIVE["train"].sum()), int((1 - LIVE["train"]).sum())
W_LIVE, W_ATTACK = len(LIVE["train"]) / (2 * max(n_live, 1)), len(LIVE["train"]) / (2 * max(n_attack, 1))
print(f"train: {n_live} live, {n_attack} not_live → trọng số lớp live {W_LIVE:.2f}, not_live {W_ATTACK:.2f}")

In [ ]:
with step("6a", "Tinh chỉnh MiniFASNet V2 (toàn bộ)"):
    MINI_ONNX = onnx.load(str(PATH_MINIFAS))
    mini_model = build_torch_graph(onnx_to_ir(MINI_ONNX)).to(DEVICE)
    probe_x = np.asarray(RAW80[IDX["calib_fit"][:8]]).astype(np.float32).transpose(0, 3, 1, 2)
    with torch.no_grad():
        diff = np.abs(mini_model.eval()(torch.from_numpy(probe_x).to(DEVICE)).cpu().numpy()
                      - MINIFAS_SESSION.run(None, {MINIFAS_SESSION.get_inputs()[0].name: probe_x})[0]).max()  # fmt: skip
    print(f"  Đồ thị PyTorch khớp ONNX gốc: lệch tối đa {diff:.2e}")
    assert diff < 1e-2, "Đồ thị PyTorch không khớp MiniFASNet gốc"
    if RUN_MINIFAS:
        HISTORY["minifasnet_ft"] = train_model(
            "minifasnet_ft", mini_model, [{"params": list(mini_model.parameters()), "lr": LR_MINIFAS}],
            "minifas", EPOCHS_MINIFAS, BATCH_MINIFAS,
            lambda logits, y: binary_live_loss(logits, y, (W_LIVE, W_ATTACK)),
        )  # fmt: skip
        SCORES["minifasnet_ft"] = predict_p_live(mini_model, "minifas", IDX["calib_fit"], 512)
        SCORES_LTT["minifasnet_ft"] = predict_p_live(mini_model, "minifas", IDX["calib_ltt"], 512)
        METRICS["minifasnet_ft"] = evaluate(SCORES["minifasnet_ft"])
        show("MiniFASNet tinh chỉnh", METRICS["minifasnet_ft"])

In [ ]:
with step("6b", "Tinh chỉnh DINOv2-reg ViT-B/14 (vài khối cuối)"):
    from transformers import Dinov2WithRegistersModel

    class DinoPad(nn.Module):
        def __init__(self, backbone) -> None:
            super().__init__()
            self.backbone = backbone
            self.n_reg = int(backbone.config.num_register_tokens)
            self.head = nn.Linear(2 * backbone.config.hidden_size, 2)  # lớp 1 = thật

        def forward(self, x):
            h = self.backbone(pixel_values=x).last_hidden_state
            return self.head(torch.cat([h[:, 0], h[:, 1 + self.n_reg :].mean(dim=1)], dim=1))

    if RUN_DINO:
        backbone = Dinov2WithRegistersModel.from_pretrained(str(DINO_DIR))
        for p in backbone.parameters():
            p.requires_grad = False
        tuned = list(backbone.encoder.layer[-DINO_TRAIN_BLOCKS:]) + [backbone.layernorm]
        for module in tuned:
            for p in module.parameters():
                p.requires_grad = True
        dino_model = DinoPad(backbone).to(DEVICE)
        n_train = sum(p.numel() for p in dino_model.parameters() if p.requires_grad)
        print(f"  Tham số được dạy: {n_train / 1e6:.1f} triệu / {sum(p.numel() for p in dino_model.parameters()) / 1e6:.1f} triệu")
        class_w = torch.tensor([W_ATTACK, W_LIVE], dtype=torch.float32, device=DEVICE)
        HISTORY["dinov2_ft"] = train_model(
            "dinov2_ft", dino_model,
            [{"params": [p for m in tuned for p in m.parameters()], "lr": LR_DINO_BLOCKS},
             {"params": list(dino_model.head.parameters()), "lr": LR_DINO_HEAD}],
            "dino", EPOCHS_DINO, BATCH_DINO,
            lambda logits, y: F.cross_entropy(logits, y, weight=class_w),
        )  # fmt: skip
        SCORES["dinov2_ft"] = predict_p_live(dino_model, "dino", IDX["calib_fit"], BATCH_DINO * 2)
        SCORES_LTT["dinov2_ft"] = predict_p_live(dino_model, "dino", IDX["calib_ltt"], BATCH_DINO * 2)
        METRICS["dinov2_ft"] = evaluate(SCORES["dinov2_ft"])
        show("DINOv2 tinh chỉnh", METRICS["dinov2_ft"])

## Bước 7 — Hiệu chỉnh, gộp hai mô hình và khuyến nghị (CHỈ trên calib_fit)

ADR-0015: **giữ MiniFASNet và THÊM tầng 2 DINOv2** — hai mô hình bổ trợ nhau, không phải chọn một.
- Mỗi mô hình được **hiệu chỉnh** trên calib_fit bằng đúng `vericlue_ml.common.calibration` của hệ thống
  (Platt/isotonic, GroupKFold theo nhóm người — ADR-0010), mỗi mô hình một `model_version` riêng.
- **GỘP** = lấy xác suất tấn công đã hiệu chỉnh LỚN NHẤT của MiniFASNet (bản tốt hơn giữa gốc/tinh chỉnh)
  và DINOv2 — khớp TrustRouter `fusion.method = max`.
- Quy tắc: GỘP hơn mô hình đơn lẻ tốt nhất (APCER tại BPCER ≤ 1 % thấp hơn; hòa thì AUC cao hơn)
  → khuyến nghị **dùng cả hai**; ngược lại dùng mô hình đơn lẻ tốt nhất.

`calib_ltt` và `test` KHÔNG được nhìn ở đây.

In [ ]:
with step("7", "Hiệu chỉnh, gộp và khuyến nghị trên calib_fit"):
    Y_ATTACK = 1 - LIVE["calib_fit"]
    DET_TAG = f"scrfd-10g@{MANIFEST['scrfd-10g']['revision'][:8]}"
    MODEL_VERSIONS = {
        "baseline": f"minifasnet-v2@{MANIFEST['minifasnet']['revision'][:8]}+{DET_TAG}",  # trùng bản đang chạy
        "minifasnet_ft": f"minifasnet-v2-ft@pending+{DET_TAG}",  # bước 10 thay 'pending' bằng SHA-256 file ONNX
        "dinov2_ft": f"dinov2-reg-b14-ft@pending+{DET_TAG}",  # bước 10 thay bằng SHA-256 file trọng số
    }
    CALIBRATORS: dict = {}
    for k in ("baseline", "minifasnet_ft", "dinov2_ft"):
        if k not in SCORES:
            continue
        try:
            CALIBRATORS[k] = calibration_fit(
                CalibFit(scores=[float(v) for v in 1 - SCORES[k]], labels=[int(v) for v in Y_ATTACK],
                         subject_ids=list(GROUPS["calib_fit"]), dataset="fas_vn_regrouped_face04"),
                module="face_pad", model_version=MODEL_VERSIONS[k], commit=SOURCE_COMMIT,
            )  # fmt: skip
            c = CALIBRATORS[k]
            print(f"  Hiệu chỉnh {k}: {c.method}, log-loss {c.before.log_loss:.4f} → {c.after.log_loss:.4f} (OOF theo nhóm)")
        except CalibrationError as exc:
            print(f"  KHÔNG hiệu chỉnh được {k}: {exc} — mô hình này không vào phần gộp.")
    CAL_P = {k: c.predict(1 - SCORES[k]) for k, c in CALIBRATORS.items()}
    minis = [k for k in ("baseline", "minifasnet_ft") if k in METRICS]
    MINI_BEST = min(minis, key=lambda k: rank_key(METRICS[k])) if minis else None
    if MINI_BEST in CAL_P and "dinov2_ft" in CAL_P:
        COMBO_P = combo_max(CAL_P[MINI_BEST], CAL_P["dinov2_ft"])
        SCORES_COMBO_ATTACK = COMBO_P
        METRICS["combo"] = pad_metrics(Y_ATTACK, COMBO_P, GROUPS["calib_fit"], MAX_BPCER, N_BOOT, SEED)
    SINGLES = {k: METRICS[k] for k in ("baseline", "minifasnet_ft", "dinov2_ft") if k in METRICS}
    RECOMMENDATION = recommend(SINGLES, METRICS.get("combo"))
    RECOMMENDATION["combo_members"] = [MINI_BEST, "dinov2_ft"] if "combo" in METRICS else []
    RECOMMENDATION["rule"] = "GỘP (max xác suất đã hiệu chỉnh) thắng đơn lẻ tốt nhất theo APCER@BPCER≤1%, hòa thì AUC"
    WINNER = RECOMMENDATION["choice"]
    names = {"baseline": "MiniFASNet gốc", "minifasnet_ft": "MiniFASNet tinh chỉnh",
             "dinov2_ft": "DINOv2-reg tinh chỉnh", "combo": f"GỘP max({MINI_BEST} + DINOv2)"}  # fmt: skip
    REF = next(iter(METRICS.values()))
    print(f"Bảng so sánh trên calib_fit (live {REF['n_live']}, tấn công "
          f"{REF['n_attack']}, {REF.get('n_groups', '?')} nhóm người):")
    print(f"  {'mô hình':34s} {'AUC':>7s} {'EER':>7s} {'APCER@BPCER≤1%':>15s} {'CI95 APCER':>17s}")
    for k, m in METRICS.items():
        ci = m.get("apcer_ci95") or [float("nan")] * 2
        mark = "  ← KHUYẾN NGHỊ" if k == WINNER else ""
        print(f"  {names[k]:34s} {m['auc']:7.4f} {m['eer']:7.4f} {m['apcer']:15.4f}   [{ci[0]:.3f}–{ci[1]:.3f}]{mark}")
    if WINNER == "combo":
        print(f"\nKHUYẾN NGHỊ: DÙNG CẢ HAI — {names[MINI_BEST]} + DINOv2, gộp bằng max xác suất đã hiệu chỉnh "
              f"(khớp TrustRouter fusion.method = max). Lý do: {RECOMMENDATION['reason_vi']}.")
        print("  Việc còn mở: server cần wrapper DINOv2 như một mô-đun mặt thứ hai.")
    else:
        print(f"\nKHUYẾN NGHỊ: dùng MỘT mô hình — {names[WINNER]}. Lý do: {RECOMMENDATION['reason_vi']}")
    print("Ghi chú: GỘP dùng calibrator fit trên chính calib_fit (Platt 2 tham số) → số của GỘP hơi lạc quan.")
    BEATS_BASELINE = None if "baseline" not in METRICS else (WINNER != "baseline" and rank_key(METRICS[WINNER]) < rank_key(METRICS["baseline"]))
    if MODEL != "both":
        print(f"Lần chạy này chỉ có {MODEL}. Gộp MiniFASNet + DINOv2 do trưởng nhóm chạy bằng ml/notebooks/combine_pad.py.")

## Bước 8 — Test phụ (nếu có)

- **Axon** (ảnh in, ảnh in cắt, mặt nạ giấy 3D, phát lại màn hình/điện thoại) và **UniData Asian**
  (chỉ người thật → BPCER): tự tải mẫu nhỏ từ Hugging Face (công khai, ghim revision) khi Internet bật;
  hoặc Add Input một dataset Kaggle có tên chứa `axon` / `unique`.
- **Zalo 2022** (khẩu trang): Add Input `hlly34/liveness-detection-zalo-2022`. Thiếu thì bỏ qua.

Ngưỡng dùng ở đây là ngưỡng BPCER ≤ 1 % đã chọn trên calib_fit. Số mẫu rất nhỏ → chỉ để tham khảo.

In [ ]:
with step("8", "Đo test phụ"):
    VIDEO_EXT = {".mp4", ".mov", ".avi", ".webm", ".mkv"}
    IMAGE_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    sources: list[tuple[str, Path]] = []
    if INPUT_ROOT.is_dir():
        for d in sorted(INPUT_ROOT.iterdir()):
            low = d.name.lower()
            if d.is_dir() and ("axon" in low or "unique" in low or "unidata" in low):
                sources.append((low, d))
    for d in AUX_DIRS:
        for sub in sorted(d.iterdir()) if d.is_dir() else []:
            if sub.is_dir() and any(x in sub.name.lower() for x in ("axon", "unique", "unidata")):
                sources.append((sub.name.lower(), sub))
    if AUX_FROM_HF:
        from huggingface_hub import HfApi, hf_hub_download

        api = HfApi()
        for key, spec in AUX_HF.items():
            if any(k == key for k, _ in sources):
                continue
            try:
                files = sorted(api.list_repo_files(spec["repo"], repo_type="dataset", revision=spec["revision"]))
                files = [f for f in files if not f.startswith(".") and Path(f).suffix.lower() in VIDEO_EXT | IMAGE_EXT]
                if spec.get("prefix"):
                    files = [f for f in files if f.startswith(spec["prefix"])]
                if spec.get("video_only"):
                    files = [f for f in files if Path(f).suffix.lower() in VIDEO_EXT]
                files = [f for f in files if Path(f).name not in spec.get("exclude", [])]
                if spec.get("every"):
                    files = files[:: spec["every"]]
                if spec.get("limit"):
                    files = files[: spec["limit"]]
                if AUX_MAX_FILES:
                    files = files[:AUX_MAX_FILES]
                target = WORK / "aux" / key
                for f in files:
                    hf_hub_download(spec["repo"], f, repo_type="dataset", revision=spec["revision"], local_dir=target)
                sources.append((key, target))
                print(f"  tải {key}: {len(files)} file ({spec['repo']}@{spec['revision'][:8]}, {spec['license']})")
            except Exception as exc:  # noqa: BLE001
                print(f"  BỎ QUA {key}: không tải được ({str(exc)[:150]})")
    zalo_csv = None
    for pattern in ("*/label.csv", "*/*/label.csv", "*/*/*/label.csv"):
        for cand in sorted(INPUT_ROOT.glob(pattern)) + [c for d in AUX_DIRS for c in sorted(d.glob(pattern))]:
            with open(cand, encoding="utf-8", errors="ignore") as handle:
                if "liveness_score" in handle.readline():
                    zalo_csv = zalo_csv or cand

    def onnx_p_live_arr(session, crops):
        x = np.asarray(crops).astype(np.float32).transpose(0, 3, 1, 2)
        return _softmax(session.run(None, {session.get_inputs()[0].name: x})[0])[:, 1]
    def torch_p_live(model, kind, crops):
        model.eval()
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=(kind == "dino" and DEVICE == "cuda")):
            x = torch.stack([to_input(c, kind) for c in crops]).to(DEVICE)
            return torch.softmax(model(x).float(), dim=1)[:, 1].cpu().numpy()
    models_live: dict = {}
    if "baseline" in METRICS:
        models_live["baseline"] = lambda c80, c224: onnx_p_live_arr(MINIFAS_SESSION, c80)
    if "minifasnet_ft" in METRICS:
        models_live["minifasnet_ft"] = lambda c80, c224: torch_p_live(mini_model, "minifas", c80)
    if "dinov2_ft" in METRICS:
        models_live["dinov2_ft"] = lambda c80, c224: torch_p_live(dino_model, "dino", c224)

    def media_items():
        for key, root in sources:
            files = sorted(p for p in root.rglob("*") if p.is_file() and ".cache" not in p.parts
                           and p.suffix.lower() in VIDEO_EXT | IMAGE_EXT)  # fmt: skip
            if AUX_MAX_FILES:
                files = files[:AUX_MAX_FILES]
            for p in files:
                lab = aux_label_from_path(key, p.relative_to(root).as_posix())
                if lab:
                    yield ("unidata" if "uni" in key else "axon"), key, p, lab[0], lab[1]
        if zalo_csv is not None:
            with open(zalo_csv, newline="", encoding="utf-8") as handle:
                table = list(csv.DictReader(handle))
            by_label = {"1": [], "0": []}
            for t in table:
                by_label.setdefault(str(t["liveness_score"]).strip(), []).append(t)
            picked = table if not AUX_MAX_FILES else by_label["1"][:AUX_MAX_FILES] + by_label["0"][:AUX_MAX_FILES]
            for t in picked:
                p = zalo_csv.parent / "videos" / t["fname"]
                if p.is_file():
                    live = str(t["liveness_score"]).strip() == "1"
                    yield "zalo2022", "zalo2022", p, ("live" if live else "not_live"), ("bona_fide" if live else "replay_or_print")

    AUX_ROWS: list[dict] = []
    for dataset, key, path, label, attack in media_items():
        frames = video_frames(str(path)) if path.suffix.lower() in VIDEO_EXT else [cv2.imdecode(np.fromfile(str(path), np.uint8), cv2.IMREAD_COLOR)]
        for fi, img in enumerate(frames):
            if img is None:
                continue
            box, status, padded, _ = detect_face(FINDER, img, DET_PAD_RATIO)
            if box is None:
                AUX_ROWS.append({"dataset": dataset, "subset": key, "file": path.name, "frame": fi, "label": label, "attack": attack, "face": status})
                continue
            c80 = jpeg_roundtrip(pad_crop(img, box, MINI_SIZE), JPEG_QUALITY)[0]
            c224 = jpeg_roundtrip(pad_crop(img, box, DINO_SIZE), JPEG_QUALITY)[0]
            row = {"dataset": dataset, "subset": key, "file": path.name, "frame": fi, "label": label, "attack": attack, "face": "ok", "det_padded": int(padded)}
            for name, fn in models_live.items():
                row["attack_score_" + name] = float(1 - fn([c80], [c224])[0])
            AUX_ROWS.append(row)
    eval_names = list(models_live)
    if "combo" in METRICS:
        mb = RECOMMENDATION["combo_members"][0]
        for r in AUX_ROWS:
            if r["face"] == "ok":
                r["attack_score_combo"] = float(combo_max(CALIBRATORS[mb].predict([r["attack_score_" + mb]]),
                                                          CALIBRATORS["dinov2_ft"].predict([r["attack_score_dinov2_ft"]]))[0])  # fmt: skip
        eval_names.append("combo")
    AUX_RESULTS: dict = {}
    if not AUX_ROWS:
        print("Không có dữ liệu test phụ (chưa Add Input / Internet tắt) → BỎ QUA, ghi chú trong card.md.")
    for dataset in sorted({r["dataset"] for r in AUX_ROWS}):
        part = [r for r in AUX_ROWS if r["dataset"] == dataset and r["face"] == "ok"]
        n_noface = sum(1 for r in AUX_ROWS if r["dataset"] == dataset and r["face"] != "ok")
        res = {"n_frames": len(part), "n_files": len({(r["subset"], r["file"]) for r in part}), "n_no_face": n_noface, "models": {}}
        y = np.array([int(r["label"] == "not_live") for r in part])
        for name in eval_names:
            if not part:
                continue
            s = np.array([r["attack_score_" + name] for r in part])
            t = METRICS[name]["threshold_at_bpcer"]
            m = {"threshold_from_calib_fit": t}
            if (y == 0).any():
                m["bpcer"] = float(np.mean(s[y == 0] >= t))
            if (y == 1).any():
                m["apcer_by_attack"] = apcer_by_attack(y, s, [r["attack"] for r in part], t)
                m["n_by_attack"] = {a: sum(1 for r in part if r["attack"] == a) for a in m["apcer_by_attack"]}
            if len(set(y.tolist())) == 2:
                m["auc"] = auc(y, s)
            res["models"][name] = m
        AUX_RESULTS[dataset] = res
        print(f"\n  {dataset}: {res['n_frames']} khung từ {res['n_files']} file (không thấy mặt: {n_noface})")
        for name, m in res["models"].items():
            print(f"    {name:14s}", json.dumps({k: (round(v, 3) if isinstance(v, float) else v) for k, v in m.items() if k != 'threshold_from_calib_fit'}, ensure_ascii=False))

## Bước 9 — Kiểm cuối: không bước nào chạm split `test`

In [ ]:
with step("9", "Kiểm cuối: không chạm split test"):
    TEST_IDX = sorted(i for i, r in enumerate(rows) if r["split"] == "test")
    leaked = USED.intersection(TEST_IDX)
    assert not leaked, f"LỖI NGHIÊM TRỌNG: {len(leaked)} ảnh test đã bị đọc khi dạy/đo!"
    if TEST_IDX:
        for arr in (CROPS80, CROPS224, RAW80, RAW224):
            assert int(np.asarray(arr[TEST_IDX]).max()) == 0, "Ảnh cắt của test phải bằng 0"
    print(f"OK: {len(USED)} ảnh đã dùng (train + calib_fit + calib_ltt; calib_ltt chỉ để xuất điểm), 0 ảnh thuộc test ({len(TEST_IDX)} ảnh test chưa từng được dùng).")
    print("Trưởng nhóm sẽ chạy test MỘT lần trên máy sau khi chốt mô hình.")

## Bước 10 — Xuất kết quả (1 file zip)

Zip nằm trong thư mục output (Kaggle: tab **Output** bên phải, thư mục `/kaggle/working`).
Gồm: mô hình, SHA-256, `card.md`, `metrics.json`, `splits.csv`, điểm calib_fit, báo cáo làm sạch.

In [ ]:
with step("10", "Xuất kết quả"):
    import zipfile

    stamp = time.strftime("%Y%m%d-%H%M")
    EXP = OUT_DIR / f"pad_export_{MODEL}_{stamp}{'_smoke' if SMOKE else ''}"
    EXP.mkdir(parents=True, exist_ok=True)
    EXPORTED: dict = {}
    if "minifasnet_ft" in METRICS:
        new = write_onnx_weights(MINI_ONNX, mini_model.initializer_arrays())
        onnx.checker.check_model(new)
        onnx.save(new, str(EXP / "minifasnet_v2_ft.onnx"))
        sess = ort.InferenceSession(str(EXP / "minifasnet_v2_ft.onnx"), providers=["CPUExecutionProvider"])
        same_io = ([(i.name, i.shape) for i in sess.get_inputs()] == [(i.name, i.shape) for i in MINIFAS_SESSION.get_inputs()]
                   and [(o.name, o.shape) for o in sess.get_outputs()] == [(o.name, o.shape) for o in MINIFAS_SESSION.get_outputs()])  # fmt: skip
        check = IDX["calib_fit"][:64]
        gap = float(np.abs(onnx_p_live(sess, check) - SCORES["minifasnet_ft"][: len(check)]).max())
        clf = MiniFasNetClassifier(sess)  # đúng lớp của pad.py nạp được file mới
        print(f"  minifasnet_v2_ft.onnx: đầu vào/ra giống bản gốc = {same_io}; lệch p_live ONNX vs PyTorch = {gap:.2e}; cỡ {clf._size}")
        assert same_io and gap < 1e-3
        EXPORTED["minifasnet_ft"] = {"file": "minifasnet_v2_ft.onnx", "same_io_as_original": same_io, "max_gap_vs_torch": gap,
                                     "best_epoch": HISTORY["minifasnet_ft"]["best_epoch"]}  # fmt: skip
    if "dinov2_ft" in METRICS:
        from safetensors.torch import save_file

        save_file({k: v.contiguous() for k, v in trainable_state(dino_model).items()}, str(EXP / "dinov2_pad_delta.safetensors"))
        dino_cfg = {
            "base_model": MANIFEST["dinov2-reg-base"]["repo"],
            "base_revision": MANIFEST["dinov2-reg-base"]["revision"],
            "trained_blocks": DINO_TRAIN_BLOCKS,
            "head": "Linear(2*hidden, 2) trên [CLS ‖ trung bình patch (bỏ 4 register)]; softmax[:,1] = p_real",
            "input": f"cắt mặt như pad.py (PAD_CROP_SCALE={PAD_CROP_SCALE}) → {DINO_SIZE}x{DINO_SIZE}, BGR→RGB, /255, chuẩn hóa ImageNet",
            "load": "backbone = Dinov2WithRegistersModel.from_pretrained(base, revision=base_revision); model = DinoPad(backbone); "
                    "model.load_state_dict(safetensors.torch.load_file('dinov2_pad_delta.safetensors'), strict=False)",
            "best_epoch": HISTORY["dinov2_ft"]["best_epoch"],
        }
        (EXP / "dinov2_pad_config.json").write_text(json.dumps(dino_cfg, indent=1, ensure_ascii=False))
        EXPORTED["dinov2_ft"] = {"file": "dinov2_pad_delta.safetensors", **dino_cfg}
        if EXPORT_DINO_ONNX:
            try:
                dino_model.float().eval().cpu()
                emb = dino_model.backbone.embeddings
                probe = torch.from_numpy(np.stack([to_input(np.asarray(RAW224[i]), "dino").numpy() for i in IDX["calib_fit"][:8]]))
                with torch.no_grad():
                    ref = torch.softmax(dino_model(probe), dim=1)[:, 1].numpy()
                    n_patch = (DINO_SIZE // dino_model.backbone.config.patch_size) ** 2
                    fixed_pos = emb.interpolate_pos_encoding(torch.zeros(1, 1 + n_patch, emb.position_embeddings.shape[-1]), DINO_SIZE, DINO_SIZE).clone()
                # Nội suy bicubic-antialias của vị trí không xuất ONNX được → cố định sẵn cho cỡ 224 (chỉ chạy 224×224).
                emb.interpolate_pos_encoding = lambda embeddings, height, width: fixed_pos.to(embeddings.dtype)
                torch.onnx.export(dino_model, torch.zeros(1, 3, DINO_SIZE, DINO_SIZE), str(EXP / "dinov2_pad.onnx"),
                                  input_names=["input"], output_names=["output"], opset_version=17,
                                  dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}}, dynamo=False)  # fmt: skip
                del emb.interpolate_pos_encoding
                dsess = ort.InferenceSession(str(EXP / "dinov2_pad.onnx"), providers=["CPUExecutionProvider"])
                got = _softmax(dsess.run(None, {"input": probe.numpy()})[0])[:, 1]
                dgap = float(np.abs(got - ref).max())
                print(f"  dinov2_pad.onnx: lệch p_live ONNX vs PyTorch = {dgap:.2e} (chỉ nhận ảnh {DINO_SIZE}×{DINO_SIZE})")
                if dgap > 1e-3:
                    raise ValueError(f"ONNX lệch PyTorch {dgap:.2e}")
                EXPORTED["dinov2_ft"].update({"onnx": "dinov2_pad.onnx", "onnx_max_gap_vs_torch": dgap})
            except Exception as exc:  # noqa: BLE001
                (EXP / "dinov2_pad.onnx").unlink(missing_ok=True)
                EXPORTED["dinov2_ft"]["onnx_error"] = str(exc)[:300]
                print("  Không xuất được ONNX cho DINOv2 (vẫn có safetensors):", str(exc)[:200])

    if AUX_ROWS:
        with open(EXP / "aux_scores.csv", "w", newline="", encoding="utf-8") as handle:
            keys = sorted({k for r in AUX_ROWS for k in r})
            writer = csv.DictWriter(handle, fieldnames=keys)
            writer.writeheader()
            writer.writerows(AUX_ROWS)
    for name in list(SPLIT_FILE_SHA) + ["cleaning_report.json", "leak_check.json", "prep_done.json"]:
        shutil.copy2(PREP_DIR / name, EXP / name)

    model_sha = {f.name: sha256_file(f) for f in sorted(EXP.iterdir()) if f.suffix in (".onnx", ".safetensors")}
    # Calibrator JSON: model_version gắn SHA-256 file mô hình (đặt vào ml/configs/calibrators/face_pad/).
    from urllib.parse import quote

    final_files = {"minifasnet_ft": "minifasnet_v2_ft.onnx", "dinov2_ft": "dinov2_pad_delta.safetensors"}
    cal_dir = EXP / "calibrators"  # module = face_pad ghi trong từng JSON
    cal_dir.mkdir(parents=True, exist_ok=True)
    CALIBRATOR_FILES = {}
    for k, c in CALIBRATORS.items():
        if k in final_files and final_files[k] in model_sha:
            c = c.model_copy(update={"model_version": MODEL_VERSIONS[k].replace("@pending", "@" + model_sha[final_files[k]][:8])})
        suffix = ".fasvn-candidate.json" if k == "baseline" else ".json"  # không trùng file identity trong repo
        path = cal_dir / f"{quote(c.model_version, safe='@-_.')}{suffix}"
        path.write_text(c.model_dump_json(indent=2) + "\n", encoding="utf-8")
        CALIBRATOR_FILES[k] = {"model_version": c.model_version, "file": path.relative_to(EXP).as_posix(), "method": c.method}
    print("  Calibrator:", {k: v["file"] for k, v in CALIBRATOR_FILES.items()})
    # Điểm từng mẫu trên calib_fit và calib_ltt (KHÔNG có test): điểm thô + xác suất đã hiệu chỉnh.
    for split_name, score_map in (("calib_fit", SCORES), ("calib_ltt", SCORES_LTT)):
        with open(EXP / f"scores_{split_name}.csv", "w", newline="", encoding="utf-8") as handle:
            writer = csv.writer(handle, lineterminator="\n")
            cols = ["sample_id", "group_id", "label", "source", "split"]
            for k in score_map:
                cols += [f"raw_attack_score_{k}"] + ([f"cal_attack_prob_{k}"] if k in CALIBRATORS else [])
            writer.writerow(cols)
            cal = {k: CALIBRATORS[k].predict(1 - score_map[k]) for k in score_map if k in CALIBRATORS}
            for n, i in enumerate(IDX[split_name]):
                r = rows[i]
                assert r["split"] == split_name
                line = [r["rel_path"], r["group_id"], r["label"], r["source"], split_name]
                for k in score_map:
                    line += [f"{1 - score_map[k][n]:.6f}"] + ([f"{cal[k][n]:.6f}"] if k in cal else [])
                writer.writerow(line)
    METRICS_JSON = {
        "task": "FACE-04", "model": MODEL, "smoke": SMOKE, "env": ENV, "source_commit": SOURCE_COMMIT, "source_dirty": SOURCE_DIRTY,
        "source_files_sha256": SOURCE_FILES, "manifest": MANIFEST, "versions": VERSIONS, "config": CONFIG,
        "dataset": {"name": "FAS-VN", "kaggle": "maihongtng/face-anti-spoofing", "license_page": "CC BY 4.0 (đáng ngờ — chỉ nghiên cứu)",
                    "splits_csv_sha256": SPLITS_SHA, "split_files_sha256": SPLIT_FILE_SHA, "counts": {s: {lab: sum(1 for r in rows if r["split"] == s and r["label"] == lab)
                    for lab in ("live", "not_live")} for s in ("train", "calib_fit", "calib_ltt", "test")}},
        "cleaning": CLEAN_REPORT, "leak_check": {k: LEAK[k] for k in ("model_auc", "models", "gate_features", "per_feature", "before_per_feature", "diagnostic", "with_file_size", "by_source")},
        "calib_fit": METRICS, "history": {k: v["history"] for k, v in HISTORY.items()},
        "selection": {"checkpoint_rule": "min (APCER@BPCER<=1%, EER, -AUC) trên calib_fit", **RECOMMENDATION, "beats_baseline": BEATS_BASELINE}, "calibrators": CALIBRATOR_FILES,
        "aux": AUX_RESULTS, "exported": EXPORTED, "model_sha256": model_sha, "timings_seconds": TIMINGS,
    }  # fmt: skip
    (EXP / "metrics.json").write_text(json.dumps(METRICS_JSON, indent=1, ensure_ascii=False, default=str))

    def fmt(m, k):
        v = m.get(k)
        return "—" if v is None else f"{v:.4f}"

    lines = [
        f"# Thẻ mô hình — PAD {MODEL} tinh chỉnh trên FAS-VN ({'CHẠY THỬ, KHÔNG DÙNG' if SMOKE else 'chạy đầy đủ'})",
        "",
        f"- Task: FACE-04. Notebook `ml/notebooks/finetune_pad_{MODEL}.ipynb`, commit nguồn `{SOURCE_COMMIT}`" + (" (có sửa chưa commit)" if SOURCE_DIRTY else ""),
        f"- Chạy lúc {stamp} trên {ENV} ({VERSIONS['gpu']}); seed {SEED}; torch {VERSIONS['torch']}, onnxruntime {VERSIONS['onnxruntime']}",
        "- **Khuyến nghị:** " + (f"**DÙNG CẢ HAI** ({RECOMMENDATION['combo_members'][0]} + dinov2_ft, gộp max xác suất đã hiệu chỉnh — khớp TrustRouter fusion.method = max; cần wrapper DINOv2 ở server)" if WINNER == "combo" else f"dùng một mô hình **{WINNER}**") + f". Lý do: {RECOMMENDATION['reason_vi']}",
        "- Calibrator (fit trên calib_fit, ADR-0010): " + ", ".join(f"`{v['file']}` ({v['method']})" for v in CALIBRATOR_FILES.values()),
        "",
        "## Dữ liệu",
        f"- FAS-VN (`maihongtng/face-anti-spoofing`), {CLEAN_REPORT['n_images']} ảnh đọc, giữ {CLEAN_REPORT['n_kept']} sau làm sạch; bỏ: {CLEAN_REPORT['dropped']}",
        f"- Gom người: {CLEAN_REPORT['n_groups']} nhóm (dHash ≤ {DHASH_LINK} hoặc ArcFace cos ≥ {ARC_THRESHOLD}); nhóm lớn nhất {CLEAN_REPORT['largest_group']} ảnh",
        f"- Chia lại theo nhóm (seed {SEED}): " + ", ".join(f"{s} {sum(METRICS_JSON['dataset']['counts'][s].values())}" for s in ("train", "calib_fit", "calib_ltt", "test")),
        f"- Lần chạy này: **{MODEL}**. `splits.csv` SHA-256 `{SPLITS_SHA}` (phải trùng với lần chạy mô hình kia). Split `test` CHƯA được dùng.",
        f"- Khử lộ đề: cắt mặt như pad.py → resize → JPEG q{JPEG_QUALITY}, không EXIF. AUC mô hình chỉ siêu dữ liệu = {LEAK['model_auc']:.3f} (ngưỡng ≤ {LEAK_AUC_MAX}); thêm cỡ file JPEG (tín hiệu nội dung) = {LEAK['with_file_size']['model_auc']:.3f}",
        "",
        "## Kết quả trên calib_fit (chọn mô hình chỉ ở đây)",
        "| Mô hình | AUC | EER | APCER @ BPCER ≤ 1 % | CI95 APCER (bootstrap theo nhóm) | Ngưỡng |",
        "|---|---|---|---|---|---|",
    ]
    for k, m in METRICS.items():
        ci = m.get("apcer_ci95")
        lines.append(f"| {k} | {fmt(m, 'auc')} | {fmt(m, 'eer')} | {fmt(m, 'apcer')} | {'—' if not ci else f'{ci[0]:.3f}–{ci[1]:.3f}'} | {fmt(m, 'threshold_at_bpcer')} |")
    lines += ["", f"Số mẫu calib_fit: live {REF['n_live']}, tấn công {REF['n_attack']}, {REF.get('n_groups', '?')} nhóm người.", ""]
    lines += ["## Test phụ (tham khảo, n nhỏ)", ""]
    if AUX_RESULTS:
        for d, res in AUX_RESULTS.items():
            lines.append(f"- {d}: {res['n_frames']} khung / {res['n_files']} file — " + json.dumps(res["models"], ensure_ascii=False, default=float))
    else:
        lines.append("- Không chạy (chưa có dữ liệu test phụ).")
    lines += [
        "",
        "## Tham số",
        f"- (a) MiniFASNet V2: dạy toàn bộ, AdamW lr {LR_MINIFAS}, {EPOCHS_MINIFAS} epoch, batch {BATCH_MINIFAS}; đầu vào/ra giữ nguyên ONNX gốc.",
        f"- (b) DINOv2-reg ViT-B/14 `{MANIFEST['dinov2-reg-base']['repo']}@{MANIFEST['dinov2-reg-base']['revision'][:8]}`: dạy {DINO_TRAIN_BLOCKS} khối cuối + layernorm + đầu tuyến tính, lr {LR_DINO_BLOCKS}/{LR_DINO_HEAD}, {EPOCHS_DINO} epoch, batch {BATCH_DINO}, fp16.",
        "- Tăng cường giống camera trước điện thoại rẻ cho cả 2 lớp: tối, nhiễu, mờ, thu nhỏ, JPEG 30–95, lật.",
        "",
        "## Giới hạn (bắt buộc đọc)",
        "- FAS-VN gần như **chỉ có tấn công chụp lại màn hình**; **thiếu ảnh in giấy, ảnh in cắt, mặt nạ** → mô hình chưa được dạy các kiểu này (xem Axon ở test phụ).",
        "- Giấy phép FAS-VN đáng ngờ (trang ghi CC BY 4.0 nhưng ảnh có vẻ từ hệ thống eKYC thật): **chỉ dùng nghiên cứu**, không phân phối lại, không đưa mặt người lên báo cáo/demo công khai.",
        "- Ảnh live nguồn A chủ yếu chụp tại quầy, không phải selfie; nguồn ảnh tương quan với nhãn (xem `leak_check.json` → diagnostic).",
    ]
    bs = LEAK["by_source"]
    lines.append(
        "- AUC cỡ file JPEG (độ nét) gộp / trong nguồn A / trong nguồn B: "
        + " / ".join(f"{bs[k]['file_size_auc']:.3f}" if "file_size_auc" in bs.get(k, {}) else "—" for k in ("pooled", "A", "B"))
        + "; AUC cổng siêu dữ liệu: "
        + " / ".join(f"{bs[k]['gate_auc']:.3f}" if "gate_auc" in bs.get(k, {}) else "—" for k in ("pooled", "A", "B"))
    )
    if bs["source_driven_file_size"]:
        lines.append("- **CẢNH BÁO: tín hiệu cỡ file/độ nét do nguồn ảnh (A/B) gây ra**, trong từng nguồn ≈ 0,5 — mô hình có thể học nguồn thay vì thật/giả.")
    lines += [
        "- Nhóm người suy từ ArcFace + dHash (không có mã người thật); có thể còn sót người trùng giữa các split.",
        "- SCRFD `det_10g` và ArcFace `arcface-mnet`: giấy phép nghiên cứu phi thương mại.",
        "- Số đo calib_fit KHÔNG phải kết quả test. Kết quả R1 chỉ có sau khi trưởng nhóm chạy split `test` một lần.",
    ]
    if SMOKE:
        lines.insert(2, "> **CHẠY THỬ (SMOKE)**: vài trăm ảnh, 1 epoch — số liệu không có giá trị, chỉ để kiểm notebook chạy được.\n")
    (EXP / "card.md").write_text("\n".join(lines) + "\n", encoding="utf-8")

    sums = [f"{sha256_file(f)}  {f.relative_to(EXP).as_posix()}" for f in sorted(EXP.rglob("*")) if f.is_file() and f.name != "SHA256SUMS.txt"]
    (EXP / "SHA256SUMS.txt").write_text("\n".join(sums) + "\n")
    ZIP = OUT_DIR / f"{EXP.name}.zip"
    with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in sorted(EXP.rglob("*")):
            if f.is_file():
                zf.write(f, f"{EXP.name}/{f.relative_to(EXP).as_posix()}")
    print(f"\nXONG. File cần gửi trưởng nhóm: {ZIP} ({ZIP.stat().st_size / 1e6:.1f} MB)")
    print("Kaggle: mở tab Output (cột phải) → bấm vào file .zip → Download. Colab: file nằm trong Google Drive/vericlue_pad.")
    print("Thời gian từng bước (phút):", {k: round(v / 60, 1) for k, v in TIMINGS.items()})